In [ ]:
import os
import random
import torch
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms

In [ ]:
import numpy as np

In [ ]:
root = '/kaggle/input/apple-leaf-disease/ATLDSD'
counts = {cls: len(os.listdir(os.path.join(root, cls, 'image')))
          for cls in os.listdir(root) if os.path.isdir(os.path.join(root, cls, 'image'))}
print(counts)

In [ ]:
classes = [c for c in os.listdir(root) if os.path.isdir(os.path.join(root, c, 'image'))]
fig, axes = plt.subplots(len(classes), 2, figsize=(8, 4 * len(classes)))
for i, cls in enumerate(classes):
    fname = sorted(os.listdir(os.path.join(root, cls, 'image')))[0]
    lbl_fname = os.path.splitext(fname)[0] + '.png'
    axes[i, 0].imshow(Image.open(os.path.join(root, cls, 'image', fname)))
    axes[i, 0].set_title(f'{cls} - Image')
    axes[i, 1].imshow(Image.open(os.path.join(root, cls, 'label', lbl_fname)), cmap='gray')
    axes[i, 1].set_title(f'{cls} - Label')
    axes[i, 0].axis('off'); axes[i, 1].axis('off')
plt.tight_layout()
for ext in ['png', 'svg', 'pdf']:
    fig.savefig(f'dataset_sam.{ext}', dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
!pip install albumentations

In [ ]:
class LeafDataset(Dataset):
    def __init__(self, root, img_size=256, augment=False):
        self.images, self.masks = [], []
        self.augment = augment
        self.img_size = img_size
        self.normalize = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        for cls in os.listdir(root):
            img_dir = os.path.join(root, cls, 'image')
            lbl_dir = os.path.join(root, cls, 'label')
            if not os.path.isdir(img_dir): continue
            for f in sorted(os.listdir(img_dir)):
                lbl_f = os.path.splitext(f)[0] + '.png'
                lbl_path = os.path.join(lbl_dir, lbl_f)
                if os.path.exists(lbl_path):
                    self.images.append(os.path.join(img_dir, f))
                    self.masks.append(lbl_path)

    def __len__(self): return len(self.images)

    def __getitem__(self, i):
        img = Image.open(self.images[i]).convert('RGB')
        mask = Image.open(self.masks[i]).convert('L')

        # Resize both
        img = transforms.functional.resize(img, (self.img_size, self.img_size))
        mask = transforms.functional.resize(mask, (self.img_size, self.img_size), interpolation=transforms.InterpolationMode.NEAREST)
        if self.augment:
            # Horizontal flip
            if random.random() > 0.5:
                img = transforms.functional.hflip(img)
                mask = transforms.functional.hflip(mask)
            # Vertical flip
            if random.random() > 0.5:
                img = transforms.functional.vflip(img)
                mask = transforms.functional.vflip(mask)
            # Random rotation
            angle = random.uniform(-30, 30)
            img = transforms.functional.rotate(img, angle)
            mask = transforms.functional.rotate(mask, angle)
            # Color jitter (image only, not mask)
            img = transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1)(img)
        img = transforms.functional.to_tensor(img)
        mask = transforms.functional.to_tensor(mask)
        # Normalize image
        img = self.normalize(img)
        mask = (mask > 0).float()

        return img, mask

In [ ]:
full_ds = LeafDataset(root, img_size=256, augment=False)

# Check per-class coverage
for cls in sorted(os.listdir(root)):
    img_dir = os.path.join(root, cls, 'image')
    lbl_dir = os.path.join(root, cls, 'label')
    if not os.path.isdir(lbl_dir): continue
    
    f = sorted(os.listdir(lbl_dir))[0]
    mask_rgb = np.array(Image.open(os.path.join(lbl_dir, f)).convert('RGB'))
    is_bg = np.all(mask_rgb == [0, 0, 0], axis=2)
    is_leaf = np.all(mask_rgb == [128, 0, 0], axis=2)
    lesion = (~is_bg & ~is_leaf)
    
    print(f"{cls:30s} → Lesion: {lesion.mean()*100:.1f}% | Leaf: {is_leaf.mean()*100:.1f}% | BG: {is_bg.mean()*100:.1f}%")

In [ ]:
from PIL import Image
import numpy as np
import random
import torch
from torchvision import transforms
import matplotlib.pyplot as plt

mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

ds_no_aug = LeafDataset(root, img_size=256, augment=False)

# Get one sample per class with lesion
class_indices = {}
for idx in range(len(ds_no_aug)):
    mask_path = ds_no_aug.masks[idx]
    cls_name = mask_path.split('/')[-3]
    if cls_name not in class_indices:
        rgb_mask = np.array(Image.open(mask_path).convert('RGB'))
        is_bg = np.all(rgb_mask == [0, 0, 0], axis=2)
        is_leaf = np.all(rgb_mask == [128, 0, 0], axis=2)
        is_lesion = ~is_bg & ~is_leaf
        if is_lesion.sum() > 100:
            class_indices[cls_name] = idx
    if len(class_indices) >= 5:
        break

fig, axes = plt.subplots(len(class_indices), 4, figsize=(20, 5 * len(class_indices)))
columns = ['Original Image', 'Original Mask', 'Augmented Image', 'Lesion Only (Augmented)']

for col, title in enumerate(columns):
    axes[0, col].set_title(title, fontsize=22, fontweight='bold')

for row, (cls_name, idx) in enumerate(sorted(class_indices.items())):
    # Load raw image and mask
    img_pil = Image.open(ds_no_aug.images[idx]).convert('RGB')
    mask_rgb_raw = np.array(Image.open(ds_no_aug.masks[idx]).convert('RGB'))

    # Resize
    img_pil = transforms.functional.resize(img_pil, (256, 256))
    mask_rgb_pil = Image.fromarray(mask_rgb_raw)
    mask_rgb_pil = transforms.functional.resize(mask_rgb_pil, (256, 256),
                                                 interpolation=transforms.InterpolationMode.NEAREST)

    # Extract lesion mask from RGB
    mask_rgb_np = np.array(mask_rgb_pil)
    is_bg = np.all(mask_rgb_np == [0, 0, 0], axis=2)
    is_leaf = np.all(mask_rgb_np == [128, 0, 0], axis=2)
    is_lesion_orig = ~is_bg & ~is_leaf

    # Create lesion-only PIL mask for augmentation
    lesion_mask_pil = Image.fromarray((is_lesion_orig.astype(np.uint8)) * 255)

    # Apply SAME random augmentation to image, RGB mask, and lesion mask
    do_hflip = random.random() > 0.5
    do_vflip = random.random() > 0.5
    angle = random.uniform(-30, 30)

    img_aug = img_pil.copy()
    mask_aug = mask_rgb_pil.copy()
    lesion_aug = lesion_mask_pil.copy()

    if do_hflip:
        img_aug = transforms.functional.hflip(img_aug)
        mask_aug = transforms.functional.hflip(mask_aug)
        lesion_aug = transforms.functional.hflip(lesion_aug)
    if do_vflip:
        img_aug = transforms.functional.vflip(img_aug)
        mask_aug = transforms.functional.vflip(mask_aug)
        lesion_aug = transforms.functional.vflip(lesion_aug)

    img_aug = transforms.functional.rotate(img_aug, angle)
    mask_aug = transforms.functional.rotate(mask_aug, angle)
    lesion_aug = transforms.functional.rotate(lesion_aug, angle)

    # Color jitter on image only
    img_aug = transforms.ColorJitter(brightness=0.3, contrast=0.3,
                                      saturation=0.3, hue=0.1)(img_aug)

    # Convert to numpy
    img_orig_np = np.array(img_pil) / 255.0
    img_aug_np = np.array(img_aug) / 255.0
    lesion_aug_np = np.array(lesion_aug) > 128

    # Original mask display (red=leaf, yellow=lesion)
    display_mask = np.ones_like(img_orig_np) * 0.15
    display_mask[is_leaf] = [0.8, 0.1, 0.1]
    display_mask[is_lesion_orig] = [1.0, 0.9, 0.0]

    # Lesion overlay on augmented image
    lesion_overlay = img_aug_np.copy()
    lesion_overlay[lesion_aug_np] = lesion_overlay[lesion_aug_np] * 0.3 + np.array([1.0, 0.9, 0.0]) * 0.7

    # Col 0: Original Image
    axes[row, 0].imshow(img_orig_np)
    axes[row, 0].set_ylabel(cls_name, fontsize=14, fontweight='bold')
    axes[row, 0].axis('off')

    # Col 1: Original RGB Mask
    axes[row, 1].imshow(display_mask)
    axes[row, 1].set_xlabel(f'Leaf: {is_leaf.mean()*100:.1f}% | Lesion: {is_lesion_orig.mean()*100:.1f}%',
                            fontsize=18, fontweight='bold')
    axes[row, 1].set_xticks([]); axes[row, 1].set_yticks([])

    # Col 2: Augmented Image
    axes[row, 2].imshow(img_aug_np)
    axes[row, 2].axis('off')

    # Col 3: Lesion Only on Augmented Image
    axes[row, 3].imshow(lesion_overlay)
    axes[row, 3].set_xlabel(f'Lesion: {lesion_aug_np.mean()*100:.1f}%',
                            fontsize=18, fontweight='bold', color='red')
    axes[row, 3].set_xticks([]); axes[row, 3].set_yticks([])

plt.tight_layout()
for ext in ['png', 'svg', 'pdf']:
    fig.savefig(f'augmentation_lesion_samples.{ext}', dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(5, 4, figsize=(20, 25))
columns = ['Image', 'RGB Mask', 'Whole Leaf', 'Lesion Only']

for col, title in enumerate(columns):
    axes[0, col].set_title(title, fontsize=22, fontweight='bold')

row = 0
for cls in sorted(os.listdir(root)):
    img_dir = os.path.join(root, cls, 'image')
    lbl_dir = os.path.join(root, cls, 'label')
    if not os.path.isdir(img_dir): continue
    
    fname = sorted(os.listdir(img_dir))[0]
    lbl_fname = os.path.splitext(fname)[0] + '.png'
    
    img = np.array(Image.open(os.path.join(img_dir, fname)).convert('RGB'))
    mask_rgb = np.array(Image.open(os.path.join(lbl_dir, lbl_fname)).convert('RGB'))
    
    old_mask = (np.any(mask_rgb != [0, 0, 0], axis=2)).astype(float)
    
    is_bg = np.all(mask_rgb == [0, 0, 0], axis=2)
    is_leaf = np.all(mask_rgb == [128, 0, 0], axis=2)
    new_mask = (~is_bg & ~is_leaf).astype(float)
    
    axes[row, 0].imshow(img)
    axes[row, 0].set_ylabel(cls, fontsize=14, fontweight='bold')
    axes[row, 0].axis('off')
    
    axes[row, 1].imshow(mask_rgb)
    axes[row, 1].axis('off')
    
    axes[row, 2].imshow(old_mask, cmap='Reds')
    axes[row, 2].set_xlabel(f'{old_mask.mean()*100:.1f}%', fontsize=20, fontweight='bold', color='Green')
    axes[row, 2].set_xticks([]); axes[row, 2].set_yticks([])
    
    axes[row, 3].imshow(new_mask, cmap='YlOrRd')
    axes[row, 3].set_xlabel(f'{new_mask.mean()*100:.1f}%', fontsize=20, fontweight='bold', color='red')
    axes[row, 3].set_xticks([]); axes[row, 3].set_yticks([])
    
    row += 1

plt.tight_layout()
for ext in ['png', 'svg', 'pdf']:
    fig.savefig(f'dataset_samples.{ext}', dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import os

classes = sorted([c for c in os.listdir(root) if os.path.isdir(os.path.join(root, c, 'image'))])

# Count original samples per class
orig_counts = {cls: len(os.listdir(os.path.join(root, cls, 'image'))) for cls in classes}

# After augmentation = same count since augmentation is online (applied per epoch)
# But effective samples double due to flips/rotations
# If you want to show augmented = 2x or 3x, adjust multiplier
aug_multiplier = 3  # each image produces ~2 augmented versions on average
aug_counts = {cls: orig_counts[cls] * aug_multiplier for cls in classes}

x = range(len(classes))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar([i - width/2 for i in x], orig_counts.values(), width, label='Before Augmentation', color='#B3CDE3', edgecolor='black')
bars2 = ax.bar([i + width/2 for i in x], aug_counts.values(), width, label='After Augmentation', color='#B5D8B0', edgecolor='black')

# Add count labels on bars
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, 
            int(bar.get_height()), ha='center', va='bottom', fontsize=14)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, 
            int(bar.get_height()), ha='center', va='bottom', fontsize=14)

ax.set_xlabel('Dataset Classes', fontsize=18, fontweight='bold')
ax.set_ylabel('Number of Samples', fontsize=18, fontweight='bold')
ax.set_title('Dataset Distribution: Before vs After Augmentation', fontsize=22, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(classes, rotation=10, ha='center', fontsize=14, fontweight='bold')
ax.legend(fontsize=13)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
for ext in ['png', 'svg', 'pdf']:
    fig.savefig(f'augmentation_distribution.{ext}', dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
train_size = int(0.8 * len(full_ds))
test_size = len(full_ds) - train_size

generator = torch.Generator().manual_seed(42)
train_indices, test_indices = random_split(range(len(full_ds)), [train_size, test_size], generator=generator)

train_ds = LeafDataset(root, img_size=256, augment=True)
test_ds = LeafDataset(root, img_size=256, augment=False)
train_ds = torch.utils.data.Subset(train_ds, train_indices.indices)
test_ds = torch.utils.data.Subset(test_ds, test_indices.indices)
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=8, shuffle=False, num_workers=2)

print(f'Train: {len(train_ds)}, Test: {len(test_ds)}')

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = ConvBlock(3, 64)
        self.enc2 = ConvBlock(64, 128)
        self.enc3 = ConvBlock(128, 256)
        self.enc4 = ConvBlock(256, 512)
        self.bottleneck = ConvBlock(512, 1024)
        self.pool = nn.MaxPool2d(2)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        bn = self.bottleneck(self.pool(e4))
        return e1, e2, e3, e4, bn


class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.up4 = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.dec4 = ConvBlock(1024, 512)
        self.up3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3 = ConvBlock(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = ConvBlock(256, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = ConvBlock(128, 64)

    def forward(self, bn, e1, e2, e3, e4):
        d4 = self.dec4(torch.cat([self.up4(bn), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return d1


class ProjectionHead(nn.Module):
    def __init__(self, in_dim=64, embed_dim=64):
        super().__init__()
        self.proj = nn.Conv2d(in_dim, embed_dim, kernel_size=1)

    def forward(self, x):
        z = self.proj(x)
        z = F.normalize(z, dim=1)
        return z


class PrototypeSimilarity(nn.Module):
    """
    Binary prototype similarity.
    2 prototypes: background + lesion.
    Each pixel is classified based on cosine similarity
    to learned disease prototype vs background prototype.
    """
    def __init__(self, embed_dim=64):
        super().__init__()
        self.embed_dim = embed_dim
        # 2 prototypes: background (0) and lesion (1)
        self.prototypes = nn.Parameter(torch.randn(2, embed_dim))

    def forward(self, z):
        B, C, H, W = z.shape
        prototypes = F.normalize(self.prototypes, dim=1)       # [2, C]
        z_flat = z.view(B, C, -1).permute(0, 2, 1)            # [B, HW, C]
        sim = torch.matmul(z_flat, prototypes.t())             # [B, HW, 2]
        sim = sim.permute(0, 2, 1).view(B, 2, H, W)           # [B, 2, H, W]

        # Return lesion probability (channel 1 vs channel 0)
        # prob = F.sigmoid(sim * 10.0, dim=1)                    # temperature scaled
        prob = F.softmax(sim * 10.0, dim=1)                    # temperature scaled
        lesion_prob = prob[:, 1:2, :, :]                       # [B, 1, H, W]
        return lesion_prob


class PSPENet(nn.Module):
    def __init__(self, embed_dim=64):
        super().__init__()
        self.encoder = Encoder()
        self.decoder = Decoder()
        self.projection = ProjectionHead(64, embed_dim)
        self.prototype_similarity = PrototypeSimilarity(embed_dim)

    def forward(self, x):
        e1, e2, e3, e4, bn = self.encoder(x)
        decoded = self.decoder(bn, e1, e2, e3, e4)
        z = self.projection(decoded)
        out = self.prototype_similarity(z)       # [B, 1, H, W]
        return out


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = PSPENet().to(device)
print(f'Device     : {device}')
print(f'Parameters : {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, pred, target):
        pred = pred.reshape(-1)
        target = target.reshape(-1)
        intersection = (pred * target).sum()
        return 1 - (2. * intersection + self.smooth) / \
                    (pred.sum() + target.sum() + self.smooth)


class CombinedLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCELoss()
        self.dice = DiceLoss()

    def forward(self, pred, target):
        return self.bce(pred, target) + self.dice(pred, target)


def iou_score(pred, target, threshold=0.5):
    pred = (pred > threshold).float()
    intersection = (pred * target).sum()
    union = pred.sum() + target.sum() - intersection
    return (intersection + 1e-8) / (union + 1e-8)


def dice_score(pred, target, threshold=0.5):
    pred = (pred > threshold).float()
    intersection = (pred * target).sum()
    return (2. * intersection + 1e-8) / (pred.sum() + target.sum() + 1e-8)


criterion = CombinedLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=5, factor=0.5
)

In [ ]:
imgs, masks = next(iter(train_loader))

# masks should be [B, 1, H, W] float for BCE
if masks.dim() == 3:
    masks = masks.unsqueeze(1)          # [B, H, W] → [B, 1, H, W]
masks = masks.float()

print(f'Image shape  : {imgs.shape}')
print(f'Mask shape   : {masks.shape}')
print(f'Mask unique  : {masks.unique().tolist()}')

imgs_dev = imgs.to(device)
masks_dev = masks.to(device)
out = model(imgs_dev)

print(f'Output shape : {out.shape}')           # [B, 1, H, W]
print(f'Output range : [{out.min().item():.4f}, {out.max().item():.4f}]')

loss = criterion(out, masks_dev)
print(f'Loss         : {loss.item():.4f}')
print(f'IoU          : {iou_score(out, masks_dev):.4f}')
print(f'Dice         : {dice_score(out, masks_dev):.4f}')

In [ ]:
def pixel_accuracy(preds, masks, threshold=0.5):
    """Overall Pixel Accuracy: (TP + TN) / Total"""
    preds_bin = (preds > threshold).float()
    correct = (preds_bin == masks).sum()
    return correct / masks.numel()

def precision_score(preds, masks, threshold=0.5, smooth=1e-6):
    """Precision: TP / (TP + FP)"""
    preds_bin = (preds > threshold).float()
    tp = (preds_bin * masks).sum()
    fp = (preds_bin * (1 - masks)).sum()
    return (tp + smooth) / (tp + fp + smooth)

def recall_score(preds, masks, threshold=0.5, smooth=1e-6):
    """Recall: TP / (TP + FN)"""
    preds_bin = (preds > threshold).float()
    tp = (preds_bin * masks).sum()
    fn = ((1 - preds_bin) * masks).sum()
    return (tp + smooth) / (tp + fn + smooth)

def f1_score(preds, masks, threshold=0.5, smooth=1e-6):
    """F1 = Dice = 2*TP / (2*TP + FP + FN)"""
    prec = precision_score(preds, masks, threshold, smooth)
    rec = recall_score(preds, masks, threshold, smooth)
    return (2 * prec * rec + smooth) / (prec + rec + smooth)

def iou_score(preds, masks, threshold=0.5, smooth=1e-6):
    """IoU for foreground class"""
    preds_bin = (preds > threshold).float()
    intersection = (preds_bin * masks).sum()
    union = preds_bin.sum() + masks.sum() - intersection
    return (intersection + smooth) / (union + smooth)

def dice_score(preds, masks, threshold=0.5, smooth=1e-6):
    """Dice Score (same as F1)"""
    preds_bin = (preds > threshold).float()
    intersection = (preds_bin * masks).sum()
    return (2. * intersection + smooth) / (preds_bin.sum() + masks.sum() + smooth)

def mean_iou(preds, masks, threshold=0.5, smooth=1e-6):
    """mIoU: Mean of (IoU_background + IoU_foreground) / 2"""
    preds_bin = (preds > threshold).float()
    
    # Foreground IoU (class = 1)
    inter_fg = (preds_bin * masks).sum()
    union_fg = preds_bin.sum() + masks.sum() - inter_fg
    iou_fg = (inter_fg + smooth) / (union_fg + smooth)
    
    # Background IoU (class = 0)
    preds_bg = 1 - preds_bin
    masks_bg = 1 - masks
    inter_bg = (preds_bg * masks_bg).sum()
    union_bg = preds_bg.sum() + masks_bg.sum() - inter_bg
    iou_bg = (inter_bg + smooth) / (union_bg + smooth)
    
    return (iou_fg + iou_bg) / 2.0

def mean_pixel_accuracy(preds, masks, threshold=0.5, smooth=1e-6):
    """mPA: Mean of per-class pixel accuracy = (Recall_bg + Recall_fg) / 2"""
    preds_bin = (preds > threshold).float()
    
    # Foreground accuracy (Recall for class 1)
    tp = (preds_bin * masks).sum()
    fn = ((1 - preds_bin) * masks).sum()
    acc_fg = (tp + smooth) / (tp + fn + smooth)
    
    # Background accuracy (Recall for class 0)
    tn = ((1 - preds_bin) * (1 - masks)).sum()
    fp = (preds_bin * (1 - masks)).sum()
    acc_bg = (tn + smooth) / (tn + fp + smooth)
    
    return (acc_fg + acc_bg) / 2.0

In [ ]:
num_epochs = 100
best_test_iou = 0.0

# Storage for all metrics
train_losses, test_losses = [], []
train_accs, test_accs = [], []
train_precisions, test_precisions = [], []
train_recalls, test_recalls = [], []
train_f1s, test_f1s = [], []
train_ious, test_ious = [], []
train_dices, test_dices = [], []
train_mious, test_mious = [], []
train_mpas, test_mpas = [], []

for epoch in range(num_epochs):
    # ============ TRAINING ============
    model.train()
    r_loss, r_acc, r_prec, r_rec, r_f1 = 0.0, 0.0, 0.0, 0.0, 0.0
    r_iou, r_dice, r_miou, r_mpa = 0.0, 0.0, 0.0, 0.0
    total_train = 0

    for imgs, masks in train_loader:
        imgs, masks = imgs.to(device), masks.to(device)
        preds = model(imgs)
        loss = criterion(preds, masks)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        bs = imgs.size(0)
        r_loss += loss.item() * bs
        r_acc  += pixel_accuracy(preds, masks).item() * bs
        r_prec += precision_score(preds, masks).item() * bs
        r_rec  += recall_score(preds, masks).item() * bs
        r_f1   += f1_score(preds, masks).item() * bs
        r_iou  += iou_score(preds, masks).item() * bs
        r_dice += dice_score(preds, masks).item() * bs
        r_miou += mean_iou(preds, masks).item() * bs
        r_mpa  += mean_pixel_accuracy(preds, masks).item() * bs
        total_train += bs

    train_losses.append(r_loss / total_train)
    train_accs.append(r_acc / total_train)
    train_precisions.append(r_prec / total_train)
    train_recalls.append(r_rec / total_train)
    train_f1s.append(r_f1 / total_train)
    train_ious.append(r_iou / total_train)
    train_dices.append(r_dice / total_train)
    train_mious.append(r_miou / total_train)
    train_mpas.append(r_mpa / total_train)

    # ============ TESTING ============
    model.eval()
    r_loss, r_acc, r_prec, r_rec, r_f1 = 0.0, 0.0, 0.0, 0.0, 0.0
    r_iou, r_dice, r_miou, r_mpa = 0.0, 0.0, 0.0, 0.0
    total_test = 0

    with torch.no_grad():
        for imgs, masks in test_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            preds = model(imgs)
            loss = criterion(preds, masks)

            bs = imgs.size(0)
            r_loss += loss.item() * bs
            r_acc  += pixel_accuracy(preds, masks).item() * bs
            r_prec += precision_score(preds, masks).item() * bs
            r_rec  += recall_score(preds, masks).item() * bs
            r_f1   += f1_score(preds, masks).item() * bs
            r_iou  += iou_score(preds, masks).item() * bs
            r_dice += dice_score(preds, masks).item() * bs
            r_miou += mean_iou(preds, masks).item() * bs
            r_mpa  += mean_pixel_accuracy(preds, masks).item() * bs
            total_test += bs

    test_losses.append(r_loss / total_test)
    test_accs.append(r_acc / total_test)
    test_precisions.append(r_prec / total_test)
    test_recalls.append(r_rec / total_test)
    test_f1s.append(r_f1 / total_test)
    test_ious.append(r_iou / total_test)
    test_dices.append(r_dice / total_test)
    test_mious.append(r_miou / total_test)
    test_mpas.append(r_mpa / total_test)

    scheduler.step(test_losses[-1])

    if test_ious[-1] > best_test_iou:
        best_test_iou = test_ious[-1]
        torch.save(model.state_dict(), 'best_model.pth')
        print(f'  ✅ Best model saved (IoU: {best_test_iou:.4f})')

    print(f'Epoch [{epoch+1}/{num_epochs}]')
    print(f'  Train → Loss: {train_losses[-1]:.4f} | Acc: {train_accs[-1]:.4f} | '
          f'Prec: {train_precisions[-1]:.4f} | Rec: {train_recalls[-1]:.4f} | '
          f'F1: {train_f1s[-1]:.4f} | IoU: {train_ious[-1]:.4f} | '
          f'mIoU: {train_mious[-1]:.4f} | mPA: {train_mpas[-1]:.4f}')
    print(f'  Test  → Loss: {test_losses[-1]:.4f} | Acc: {test_accs[-1]:.4f} | '
          f'Prec: {test_precisions[-1]:.4f} | Rec: {test_recalls[-1]:.4f} | '
          f'F1: {test_f1s[-1]:.4f} | IoU: {test_ious[-1]:.4f} | '
          f'mIoU: {test_mious[-1]:.4f} | mPA: {test_mpas[-1]:.4f}')
    print('-' * 100)

In [ ]:
from sklearn.manifold import TSNE

def plot_tsne_space(model, device, dataloader):
    model.eval()
    img, mask = next(iter(dataloader))
    with torch.no_grad():
        # Extract embeddings (z) before similarity
        e1, e2, e3, e4, bn = model.encoder(img.to(device))
        z = model.projection(model.decoder(bn, e1, e2, e3, e4)) 
        
    # Get learned prototypes
    protos = model.prototype_similarity.prototypes.detach().cpu().numpy() # [2, 64]
    
    # Flatten embeddings for t-SNE (sample 500 points for speed)
    z_flat = z.permute(0,2,3,1).cpu().numpy().reshape(-1, 64)
    indices = np.random.choice(len(z_flat), 500, replace=False)
    
    tsne = TSNE(n_components=2, perplexity=30)
    reduced = tsne.fit_transform(np.vstack([z_flat[indices], protos]))
    
    plt.scatter(reduced[:-2, 0], reduced[:-2, 1], c='gray', alpha=0.3, label='Pixel Embeddings')
    plt.scatter(reduced[-2, 0], reduced[-2, 1], c='blue', s=100, marker='X', label='BG Prototype')
    plt.scatter(reduced[-1, 0], reduced[-1, 1], c='red', s=100, marker='X', label='Lesion Prototype')
    plt.legend(); plt.title("t-SNE Embedding Space Separation")
    plt.show()

plot_tsne_space(model, device, test_loader)

In [ ]:
from sklearn.manifold import TSNE

def plot_tsne_space(model, device, dataloader):
    model.eval()
    imgs, masks = next(iter(dataloader))
    imgs, masks = imgs.to(device), masks.to(device)

    with torch.no_grad():
        e1, e2, e3, e4, bn = model.encoder(imgs)
        d1 = model.decoder(bn, e1, e2, e3, e4)
        z = model.projection(d1)  # (B,64,256,256)

    # Flatten embeddings and masks
    z_flat = z.permute(0,2,3,1).reshape(-1, 64).cpu().numpy()
    mask_flat = masks.reshape(-1).cpu().numpy()

    # Sample 1000 pixels
    idx = np.random.choice(len(z_flat), 1000, replace=False)
    z_sample = z_flat[idx]
    mask_sample = mask_flat[idx]

    # Get prototypes
    protos = model.prototype_similarity.prototypes.detach().cpu().numpy()

    # Combine for t-SNE
    combined = np.vstack([z_sample, protos])

    tsne = TSNE(n_components=2, perplexity=30)
    reduced = tsne.fit_transform(combined)

    z_reduced = reduced[:-2]
    proto_reduced = reduced[-2:]

    # Plot
    plt.figure(figsize=(8,6))
    plt.scatter(z_reduced[mask_sample==0,0],
                z_reduced[mask_sample==0,1],
                c='blue', alpha=0.4, label='Background Pixels')

    plt.scatter(z_reduced[mask_sample==1,0],
                z_reduced[mask_sample==1,1],
                c='red', alpha=0.6, label='Lesion Pixels')

    # plt.scatter(proto_reduced[0,0], proto_reduced[0,1],
    #             c='cyan', s=200, marker='X', label='BG Prototype')

    # plt.scatter(proto_reduced[1,0], proto_reduced[1,1],
    #             c='darkred', s=200, marker='X', label='Lesion Prototype')

    plt.title("t-SNE: Embedding Space with Prototypes")
    plt.legend()
    plt.grid(alpha=0.2)
    plt.show()
    fig.savefig(f'tsne.png', dpi=600, bbox_inches='tight', facecolor='white', edgecolor='none')
    fig.savefig(f'tsne.svg', dpi=600, bbox_inches='tight', facecolor='white', edgecolor='none')
    fig.savefig(f'tsne.pdf', dpi=600, bbox_inches='tight', facecolor='white', edgecolor='none')

plot_tsne_space(model, device, test_loader)

In [ ]:
def evaluate_robustness(model, device, loader):
    results = {}
    modifications = {
        'Clean': lambda x: x,
        'Gaussian Noise': lambda x: x + 0.1 * torch.randn_like(x),
        'Low Light': lambda x: x * 0.3,
        'Low Res (64px)': lambda x: transforms.functional.resize(transforms.functional.resize(x, 64), 256)
    }
    
    model.eval()
    for name, func in modifications.items():
        ious = []
        with torch.no_grad():
            for imgs, masks in loader:
                preds = model(func(imgs).to(device))
                ious.append(iou_score(preds, masks.to(device)).item())
        results[name] = np.mean(ious)
        
    plt.bar(results.keys(), results.values(), color=['skyblue', 'salmon', 'gold', 'lightgreen'])
    plt.ylabel('mIoU'); plt.title('Robustness Stress Test')
    plt.show()

evaluate_robustness(model, device, test_loader)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def evaluate_robustness(model, device, loader):
    results = {}
    # Definitions for different stress conditions
    modifications = {
        'Clean (Baseline)': lambda x: x,
        'Gaussian Noise\n(σ=0.1)': lambda x: x + 0.01 * torch.randn_like(x),
        'Low Light\n(30% Brightness)': lambda x: x * 0.1,
        'Low Resolution\n(64×64 Scale)': lambda x: transforms.functional.resize(
            transforms.functional.resize(x, (64, 64)), (256, 256)
        )
    }
    
    model.eval()
    for name, func in modifications.items():
        batch_ious = []
        with torch.no_grad():
            for imgs, masks in loader:
                # Apply modification and ensure range stays [0,1] if needed
                mod_imgs = torch.clamp(func(imgs), 0, 1).to(device)
                preds = model(mod_imgs)
                batch_ious.append(iou_score(preds, masks.to(device)).item())
        results[name] = np.mean(batch_ious)
    
    # --- Professional Plotting ---
    plt.style.use('seaborn-v0_8-whitegrid')
    fig, ax = plt.subplots(figsize=(10, 6))
    
    colors = ['#2E86C1', '#E74C3C', '#F1C40F', '#27AE60']
    bars = ax.bar(results.keys(), results.values(), color=colors, edgecolor='black', alpha=0.8, width=0.6)
    
    # Add value labels on top of each bar
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.3f}',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 5),  # 5 points vertical offset
                    textcoords="offset points",
                    ha='center', va='bottom', fontweight='bold', fontsize=12)

    # Styling
    ax.set_ylim(0, 1.1)  # IoU is between 0 and 1
    ax.set_ylabel('Mean IoU (mIoU)', fontsize=14, fontweight='bold')
    ax.set_title('Model Robustness Stress Analysis', fontsize=18, fontweight='bold', pad=20)
    ax.tick_params(axis='x', labelsize=11)
    
    # Add a horizontal line for the baseline for visual comparison
    plt.axhline(y=results['Clean (Baseline)'], color='gray', linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    # Save for paper
    plt.savefig('robustness_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()

# Run the evaluation
evaluate_robustness(model, device, test_loader)

In [ ]:
def plot_failures(model, device, dataset, num_cases=3):
    model.eval()
    failures = []
    
    for i in range(len(dataset)):
        img, mask = dataset[i]
        with torch.no_grad():
            pred = model(img.unsqueeze(0).to(device))
            score = iou_score(pred, mask.unsqueeze(0).to(device)).item()
        failures.append((score, img, mask, pred.squeeze(0).cpu()))
    
    # Sort by lowest IoU
    failures.sort(key=lambda x: x[0])
    
    fig, axes = plt.subplots(num_cases, 3, figsize=(12, 4*num_cases))
    for i in range(num_cases):
        score, img, mask, pred = failures[i]
        axes[i,0].imshow(img.permute(1,2,0).numpy()) # Unnormalize if needed
        axes[i,1].imshow(mask.squeeze(), cmap='gray')
        axes[i,2].imshow(pred.squeeze() > 0.5, cmap='jet')
        axes[i,0].set_ylabel(f"IoU: {score:.4f}")
    plt.tight_layout(); plt.show()
    fig.savefig(f'failure.png', dpi=600, bbox_inches='tight', facecolor='white', edgecolor='none')
    fig.savefig(f'failure.svg', dpi=600, bbox_inches='tight', facecolor='white', edgecolor='none')
    fig.savefig(f'failure.pdf', dpi=600, bbox_inches='tight', facecolor='white', edgecolor='none')
plot_failures(model, device, test_ds)

In [ ]:
from scipy.spatial.distance import directed_hausdorff

def compute_advanced_metrics(preds, masks):
    # Convert to numpy binary
    p = (preds > 0.5).float().cpu().numpy()
    m = masks.cpu().numpy()
    
    hd_scores = []
    for i in range(p.shape[0]):
        # HD is sensitive to empty masks; compute only if lesion exists
        if np.any(m[i]) and np.any(p[i]):
            u, v = np.where(p[i,0]), np.where(m[i,0])
            hd = max(directed_hausdorff(u, v)[0], directed_hausdorff(v, u)[0])
            hd_scores.append(hd)
            
    return np.mean(hd_scores) if hd_scores else 0

# You can call this inside your test loop

In [ ]:
import torch
import matplotlib.pyplot as plt

def plot_stress_samples(model, device, dataset, idx=0):
    model.eval()
    img, mask = dataset[idx]
    
    # Define transformations
    mods = {
        'Original': lambda x: x,
        'Gaussian Noise': lambda x: torch.clamp(x + 0.1 * torch.randn_like(x), 0, 1),
        'Low Light': lambda x: x * 0.3,
        'Low Resolution': lambda x: transforms.functional.resize(
            transforms.functional.resize(x, (64, 64)), (256, 256))
    }
    
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    
    # Use unnormalization for visualization (if you used standard ImageNet normalization)
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    
    for i, (name, func) in enumerate(mods.items()):
        # Prepare input
        mod_img = func(img)
        input_tensor = mod_img.unsqueeze(0).to(device)
        
        # Predict
        with torch.no_grad():
            pred = model(input_tensor).squeeze().cpu()
        
        # Unnormalize image for display
        disp_img = mod_img * std + mean
        disp_img = torch.clamp(disp_img, 0, 1).permute(1, 2, 0).numpy()
        
        # Row 0: Modified Image
        axes[0, i].imshow(disp_img)
        axes[0, i].set_title(name, fontsize=14, fontweight='bold')
        axes[0, i].axis('off')
        
        # Row 1: Model Prediction
        # Overlay prediction on original image for clarity
        axes[1, i].imshow(disp_img)
        axes[1, i].imshow(pred > 0.5, cmap='jet', alpha=0.4) # Transparent red heatmap
        
        # Calculate IoU for this specific sample
        score = iou_score(pred.unsqueeze(0), mask.unsqueeze(0)).item()
        axes[1, i].set_xlabel(f"IoU: {score:.4f}", fontsize=12, fontweight='bold')
        axes[1, i].set_xticks([]); axes[1, i].set_yticks([])

    plt.suptitle("Model Prediction Robustness under Visual Degradation", fontsize=20, y=1.02)
    plt.tight_layout()
    plt.savefig('robustness_visual_samples.png', dpi=300, bbox_inches='tight')
    plt.show()
    fig.savefig(f'blur.png', dpi=600, bbox_inches='tight', facecolor='white', edgecolor='none')
    fig.savefig(f'blur.svg', dpi=600, bbox_inches='tight', facecolor='white', edgecolor='none')
    fig.savefig(f'blur.pdf', dpi=600, bbox_inches='tight', facecolor='white', edgecolor='none')
# Run it on a specific sample from the test set
plot_stress_samples(model, device, test_ds, idx=10) 

In [ ]:
import torch
import matplotlib.pyplot as plt
from torchvision import transforms

def plot_low_res_stress(model, device, dataset, idx=None):
    model.eval()
    
    # If no index provided, pick a random one from the dataset
    if idx is None:
        idx = np.random.randint(len(dataset))
        
    img, mask = dataset[idx]
    
    # Define Low-Res logic: Downsample to 64px then upsample back to 256px
    # This simulates a "blurry" low-quality sensor
    low_res_func = lambda x: transforms.functional.resize(
        transforms.functional.resize(x, (64, 64), interpolation=transforms.InterpolationMode.BILINEAR),
        (256, 256), interpolation=transforms.InterpolationMode.BILINEAR
    )
    
    # Transformations for comparison
    scenarios = {
        'Original (256×256)': img,
        'Low-Res (64×64 Scale)': low_res_func(img)
    }
    
    fig, axes = plt.subplots(2, 2, figsize=(10, 10))
    
    # Unnormalization constants
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    
    for i, (name, mod_img) in enumerate(scenarios.items()):
        # Model Prediction
        input_tensor = mod_img.unsqueeze(0).to(device)
        with torch.no_grad():
            pred = model(input_tensor).squeeze().cpu()
        
        # Unnormalize for display
        disp_img = torch.clamp(mod_img * std + mean, 0, 1).permute(1, 2, 0).numpy()
        
        # Col i, Row 0: Image
        axes[0, i].imshow(disp_img)
        axes[0, i].set_title(name, fontsize=14, fontweight='bold')
        axes[0, i].axis('off')
        
        # Col i, Row 1: Prediction Overlay
        axes[1, i].imshow(disp_img)
        # Use a Green contour for Ground Truth and Red for Prediction
        axes[1, i].imshow(mask.squeeze(), alpha=0.3, cmap='Greens') # GT in Green
        axes[1, i].imshow(pred > 0.5, alpha=0.4, cmap='Reds')    # Pred in Red
        
        score = iou_score(pred.unsqueeze(0), mask.unsqueeze(0)).item()
        # axes[1, i].set_title(f"IoU Score: {score:.4f}", fontsize=12, color='darkred')
        axes[1, i].axis('off')

    # plt.suptitle(f"Resolution Robustness Analysis (Sample #{idx})", fontsize=18, y=0.95)
    plt.tight_layout()
    plt.show()
    fig.savefig(f'low res.png', dpi=600, bbox_inches='tight', facecolor='white', edgecolor='none')
    fig.savefig(f'low res.svg', dpi=600, bbox_inches='tight', facecolor='white', edgecolor='none')
    fig.savefig(f'low res.pdf', dpi=600, bbox_inches='tight', facecolor='white', edgecolor='none')
# Run the plot
plot_low_res_stress(model, device, test_ds)

# Results And Ablation Study

In [ ]:
!pip install thop -q

import time
import pandas as pd
import numpy as np
from scipy.stats import wilcoxon, ttest_rel, sem
from scipy import stats as scipy_stats
from sklearn.metrics import cohen_kappa_score
from sklearn.model_selection import KFold
from thop import profile
import matplotlib.pyplot as plt
import matplotlib.cm as cm

def save_fig(fig, name):
    for ext in ['png', 'svg', 'pdf']:
        fig.savefig(f'{name}.{ext}', dpi=600, bbox_inches='tight')
    print(f'  Saved: {name}.png/svg/pdf')

In [ ]:
# ==================== 1. SimpleFCN ====================
class SimpleFCN(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(64,128,3,padding=1),nn.BatchNorm2d(128),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(128,256,3,padding=1),nn.BatchNorm2d(256),nn.ReLU(),nn.MaxPool2d(2))
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256,128,2,stride=2),nn.BatchNorm2d(128),nn.ReLU(),
            nn.ConvTranspose2d(128,64,2,stride=2), nn.BatchNorm2d(64), nn.ReLU(),
            nn.ConvTranspose2d(64,32,2,stride=2),  nn.BatchNorm2d(32), nn.ReLU(),
            nn.ConvTranspose2d(32,16,2,stride=2),  nn.BatchNorm2d(16), nn.ReLU(),
            nn.Conv2d(16,1,1), nn.Sigmoid())
    def forward(self,x): return self.decoder(self.encoder(x))

# ==================== 2. SegNet ====================
class SegNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1=nn.Sequential(nn.Conv2d(3,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU(),
                                nn.Conv2d(64,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU())
        self.enc2=nn.Sequential(nn.Conv2d(64,128,3,padding=1),nn.BatchNorm2d(128),nn.ReLU(),
                                nn.Conv2d(128,128,3,padding=1),nn.BatchNorm2d(128),nn.ReLU())
        self.enc3=nn.Sequential(nn.Conv2d(128,256,3,padding=1),nn.BatchNorm2d(256),nn.ReLU(),
                                nn.Conv2d(256,256,3,padding=1),nn.BatchNorm2d(256),nn.ReLU())
        self.enc4=nn.Sequential(nn.Conv2d(256,512,3,padding=1),nn.BatchNorm2d(512),nn.ReLU(),
                                nn.Conv2d(512,512,3,padding=1),nn.BatchNorm2d(512),nn.ReLU())
        self.pool=nn.MaxPool2d(2,return_indices=True); self.unpool=nn.MaxUnpool2d(2)
        self.dec4=nn.Sequential(nn.Conv2d(512,256,3,padding=1),nn.BatchNorm2d(256),nn.ReLU(),
                                nn.Conv2d(256,256,3,padding=1),nn.BatchNorm2d(256),nn.ReLU())
        self.dec3=nn.Sequential(nn.Conv2d(256,128,3,padding=1),nn.BatchNorm2d(128),nn.ReLU(),
                                nn.Conv2d(128,128,3,padding=1),nn.BatchNorm2d(128),nn.ReLU())
        self.dec2=nn.Sequential(nn.Conv2d(128,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU(),
                                nn.Conv2d(64,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU())
        self.dec1=nn.Sequential(nn.Conv2d(64,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU(),
                                nn.Conv2d(64,1,1),nn.Sigmoid())
    def forward(self,x):
        e1,i1=self.pool(self.enc1(x)); e2,i2=self.pool(self.enc2(e1))
        e3,i3=self.pool(self.enc3(e2)); e4,i4=self.pool(self.enc4(e3))
        d4=self.dec4(self.unpool(e4,i4)); d3=self.dec3(self.unpool(d4,i3))
        d2=self.dec2(self.unpool(d3,i2)); return self.dec1(self.unpool(d2,i1))

# ==================== 3. UNet Small (3 levels) ====================
class UNet_Small(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1=ConvBlock(3,16);  self.enc2=ConvBlock(16,32);  self.enc3=ConvBlock(32,64)
        self.bottleneck=ConvBlock(64,128); self.pool=nn.MaxPool2d(2)
        self.up3=nn.ConvTranspose2d(128,64,2,stride=2); self.dec3=ConvBlock(128,64)
        self.up2=nn.ConvTranspose2d(64,32,2,stride=2);  self.dec2=ConvBlock(64,32)
        self.up1=nn.ConvTranspose2d(32,16,2,stride=2);  self.dec1=ConvBlock(32,16)
        self.final=nn.Sequential(nn.Conv2d(16,1,1),nn.Sigmoid())
    def forward(self,x):
        e1=self.enc1(x); e2=self.enc2(self.pool(e1)); e3=self.enc3(self.pool(e2))
        b=self.bottleneck(self.pool(e3))
        d3=self.dec3(torch.cat([self.up3(b),e3],1))
        d2=self.dec2(torch.cat([self.up2(d3),e2],1))
        d1=self.dec1(torch.cat([self.up1(d2),e1],1))
        return self.final(d1)

# ==================== 4. UNet Vanilla (same depth as proposed) ====================
class UNet_Vanilla(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder=Encoder(); self.decoder=Decoder()
        self.final=nn.Sequential(nn.Conv2d(64,1,1),nn.Sigmoid())
    def forward(self,x):
        e1,e2,e3,e4,bn=self.encoder(x)
        return self.final(self.decoder(bn,e1,e2,e3,e4))

# ==================== 5. Attention UNet ====================
class AttGate(nn.Module):
    def __init__(self,g_ch,x_ch,inter):
        super().__init__()
        self.Wg=nn.Sequential(nn.Conv2d(g_ch,inter,1),nn.BatchNorm2d(inter))
        self.Wx=nn.Sequential(nn.Conv2d(x_ch,inter,1),nn.BatchNorm2d(inter))
        self.psi=nn.Sequential(nn.Conv2d(inter,1,1),nn.BatchNorm2d(1),nn.Sigmoid())
        self.relu=nn.ReLU()
    def forward(self,g,x): return x*self.psi(self.relu(self.Wg(g)+self.Wx(x)))

class AttentionUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1=ConvBlock(3,64); self.enc2=ConvBlock(64,128)
        self.enc3=ConvBlock(128,256); self.enc4=ConvBlock(256,512)
        self.bottleneck=ConvBlock(512,1024); self.pool=nn.MaxPool2d(2)
        self.up4=nn.ConvTranspose2d(1024,512,2,stride=2); self.att4=AttGate(512,512,256); self.dec4=ConvBlock(1024,512)
        self.up3=nn.ConvTranspose2d(512,256,2,stride=2);  self.att3=AttGate(256,256,128); self.dec3=ConvBlock(512,256)
        self.up2=nn.ConvTranspose2d(256,128,2,stride=2);  self.att2=AttGate(128,128,64);  self.dec2=ConvBlock(256,128)
        self.up1=nn.ConvTranspose2d(128,64,2,stride=2);   self.att1=AttGate(64,64,32);    self.dec1=ConvBlock(128,64)
        self.final=nn.Sequential(nn.Conv2d(64,1,1),nn.Sigmoid())
    def forward(self,x):
        e1=self.enc1(x); e2=self.enc2(self.pool(e1))
        e3=self.enc3(self.pool(e2)); e4=self.enc4(self.pool(e3))
        b=self.bottleneck(self.pool(e4))
        d4=self.dec4(torch.cat([self.up4(b),self.att4(self.up4(b),e4)],1))
        d3=self.dec3(torch.cat([self.up3(d4),self.att3(self.up3(d4),e3)],1))
        d2=self.dec2(torch.cat([self.up2(d3),self.att2(self.up2(d3),e2)],1))
        d1=self.dec1(torch.cat([self.up1(d2),self.att1(self.up1(d2),e1)],1))
        return self.final(d1)

# ==================== 6. Residual UNet ====================
class ResBlock(nn.Module):
    def __init__(self,in_c,out_c):
        super().__init__()
        self.conv=nn.Sequential(nn.Conv2d(in_c,out_c,3,padding=1),nn.BatchNorm2d(out_c),nn.ReLU(),
                                nn.Conv2d(out_c,out_c,3,padding=1),nn.BatchNorm2d(out_c))
        self.skip=nn.Conv2d(in_c,out_c,1) if in_c!=out_c else nn.Identity()
        self.relu=nn.ReLU()
    def forward(self,x): return self.relu(self.conv(x)+self.skip(x))

class ResUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1=ResBlock(3,64); self.enc2=ResBlock(64,128)
        self.enc3=ResBlock(128,256); self.enc4=ResBlock(256,512)
        self.bottleneck=ResBlock(512,1024); self.pool=nn.MaxPool2d(2)
        self.up4=nn.ConvTranspose2d(1024,512,2,stride=2); self.dec4=ResBlock(1024,512)
        self.up3=nn.ConvTranspose2d(512,256,2,stride=2);  self.dec3=ResBlock(512,256)
        self.up2=nn.ConvTranspose2d(256,128,2,stride=2);  self.dec2=ResBlock(256,128)
        self.up1=nn.ConvTranspose2d(128,64,2,stride=2);   self.dec1=ResBlock(128,64)
        self.final=nn.Sequential(nn.Conv2d(64,1,1),nn.Sigmoid())
    def forward(self,x):
        e1=self.enc1(x); e2=self.enc2(self.pool(e1))
        e3=self.enc3(self.pool(e2)); e4=self.enc4(self.pool(e3))
        b=self.bottleneck(self.pool(e4))
        d4=self.dec4(torch.cat([self.up4(b),e4],1))
        d3=self.dec3(torch.cat([self.up3(d4),e3],1))
        d2=self.dec2(torch.cat([self.up2(d3),e2],1))
        d1=self.dec1(torch.cat([self.up1(d2),e1],1))
        return self.final(d1)

# ==================== 7. R2UNet ====================
class RecurBlock(nn.Module):
    def __init__(self,ch,t=2):
        super().__init__()
        self.t=t
        self.conv=nn.Sequential(nn.Conv2d(ch,ch,3,padding=1),nn.BatchNorm2d(ch),nn.ReLU())
    def forward(self,x):
        x1=self.conv(x)
        for _ in range(self.t-1): x1=self.conv(x+x1)
        return x1

class R2Block(nn.Module):
    def __init__(self,in_c,out_c,t=2):
        super().__init__()
        self.conv1x1=nn.Conv2d(in_c,out_c,1)
        self.rcnn=nn.Sequential(RecurBlock(out_c,t),RecurBlock(out_c,t))
    def forward(self,x):
        x1=self.conv1x1(x); return x1+self.rcnn(x1)

class R2UNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1=R2Block(3,64); self.enc2=R2Block(64,128)
        self.enc3=R2Block(128,256); self.enc4=R2Block(256,512)
        self.bottleneck=R2Block(512,1024); self.pool=nn.MaxPool2d(2)
        self.up4=nn.ConvTranspose2d(1024,512,2,stride=2); self.dec4=R2Block(1024,512)
        self.up3=nn.ConvTranspose2d(512,256,2,stride=2);  self.dec3=R2Block(512,256)
        self.up2=nn.ConvTranspose2d(256,128,2,stride=2);  self.dec2=R2Block(256,128)
        self.up1=nn.ConvTranspose2d(128,64,2,stride=2);   self.dec1=R2Block(128,64)
        self.final=nn.Sequential(nn.Conv2d(64,1,1),nn.Sigmoid())
    def forward(self,x):
        e1=self.enc1(x); e2=self.enc2(self.pool(e1))
        e3=self.enc3(self.pool(e2)); e4=self.enc4(self.pool(e3))
        b=self.bottleneck(self.pool(e4))
        d4=self.dec4(torch.cat([self.up4(b),e4],1))
        d3=self.dec3(torch.cat([self.up3(d4),e3],1))
        d2=self.dec2(torch.cat([self.up2(d3),e2],1))
        d1=self.dec1(torch.cat([self.up1(d2),e1],1))
        return self.final(d1)

print("✅ All 7 comparison models defined")

In [ ]:
def evaluate_model(m, loader):
    m.eval()
    r = {k: 0.0 for k in ['Acc','Prec','Rec','F1','IoU','Dice','mIoU','mPA']}
    total = 0
    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(device), masks.to(device)
            preds = m(imgs)
            bs = imgs.size(0)
            r['Acc']  += pixel_accuracy(preds,masks).item()*bs
            r['Prec'] += precision_score(preds,masks).item()*bs
            r['Rec']  += recall_score(preds,masks).item()*bs
            r['F1']   += f1_score(preds,masks).item()*bs
            r['IoU']  += iou_score(preds,masks).item()*bs
            r['Dice'] += dice_score(preds,masks).item()*bs
            r['mIoU'] += mean_iou(preds,masks).item()*bs
            r['mPA']  += mean_pixel_accuracy(preds,masks).item()*bs
            total += bs
    return {k: v/total for k,v in r.items()}


def train_and_evaluate(m, train_loader, test_loader, num_epochs=50, lr=1e-3, name='model'):
    m = m.to(device)
    opt = torch.optim.Adam(m.parameters(), lr=lr)
    crit = CombinedLoss()
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', patience=5, factor=0.5)
    
    best_iou = 0.0
    t_start = time.time()
    
    for epoch in range(num_epochs):
        m.train()
        for imgs, masks in train_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            preds = m(imgs)
            loss = crit(preds, masks)
            opt.zero_grad(); loss.backward(); opt.step()
        
        metrics = evaluate_model(m, test_loader)
        sched.step(1 - metrics['IoU'])
        
        if metrics['IoU'] > best_iou:
            best_iou = metrics['IoU']
            torch.save(m.state_dict(), f'{name}.pth')
        
        if (epoch+1) % 10 == 0:
            print(f'    Epoch [{epoch+1}/{num_epochs}] IoU: {metrics["IoU"]:.4f} | Best: {best_iou:.4f}')
    
    train_time = time.time() - t_start
    m.load_state_dict(torch.load(f'{name}.pth'))
    final = evaluate_model(m, test_loader)
    final['Train_Time'] = train_time
    return m, final

print("✅ Training functions defined")

In [ ]:
COMP_EPOCHS = 10  # change if needed

ALL_MODELS = {
    'SimpleFCN':      lambda: SimpleFCN(),
    'SegNet':         lambda: SegNet(),
    'UNet-Small':     lambda: UNet_Small(),
    'UNet':           lambda: UNet_Vanilla(),
    'Attention UNet': lambda: AttentionUNet(),
    'ResUNet':        lambda: ResUNet(),
    'R2UNet':         lambda: R2UNet(),
}

all_results = {}
trained_models = {}

# ---- Proposed model (already trained) ----
model.load_state_dict(torch.load('best_model.pth'))
model.to(device)
proposed_metrics = evaluate_model(model, test_loader)

# Estimate proposed training time (time 1 epoch × 100)
model.train()
_t = time.time()
for imgs, masks in train_loader:
    imgs, masks = imgs.to(device), masks.to(device)
    preds = model(imgs); loss = criterion(preds, masks)
    optimizer.zero_grad(); loss.backward(); optimizer.step()
proposed_epoch_time = time.time() - _t
proposed_metrics['Train_Time'] = proposed_epoch_time * 100  # 100 epochs

all_results['PSPENet (Proposed)'] = proposed_metrics
model.load_state_dict(torch.load('best_model.pth'))  # reload best
trained_models['PSPENet (Proposed)'] = model

# ---- Train 7 comparison models ----
for name, model_fn in ALL_MODELS.items():
    print(f'\n{"="*60}\n  Training: {name}\n{"="*60}')
    m = model_fn()
    m, metrics = train_and_evaluate(m, train_loader, test_loader,
                                     num_epochs=COMP_EPOCHS, name=name.replace(' ','_'))
    all_results[name] = metrics
    trained_models[name] = m
    print(f'  ✅ {name} → IoU: {metrics["IoU"]:.4f} | Dice: {metrics["Dice"]:.4f}')

# ---- Different Methods Table ----
methods_df = pd.DataFrame([
    {'Method': k, 'Dataset': 'ATLDSD', 'Acc': f"{v['Acc']:.4f}", 'Prec': f"{v['Prec']:.4f}",
     'Rec': f"{v['Rec']:.4f}", 'F1': f"{v['F1']:.4f}", 'DICE': f"{v['Dice']:.4f}",
     'IOU': f"{v['IoU']:.4f}", 'mIoU': f"{v['mIoU']:.4f}", 'mPA': f"{v['mPA']:.4f}"}
    for k,v in all_results.items()
])
print("\n📊 DIFFERENT METHODS TABLE")
print(methods_df.to_string(index=False))

In [ ]:
# Ablation variants of PSPENet

# A1: Without Prototype Similarity (conv output)
class PSPENet_NoProto(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder=Encoder(); self.decoder=Decoder()
        self.projection=ProjectionHead(64,64)
        self.final=nn.Sequential(nn.Conv2d(64,1,1),nn.Sigmoid())
    def forward(self,x):
        e1,e2,e3,e4,bn=self.encoder(x)
        d=self.decoder(bn,e1,e2,e3,e4)
        return self.final(self.projection(d))

# A2: Without Projection Head
class PSPENet_NoProj(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder=Encoder(); self.decoder=Decoder()
        self.prototype_similarity=PrototypeSimilarity(64)
    def forward(self,x):
        e1,e2,e3,e4,bn=self.encoder(x)
        d=self.decoder(bn,e1,e2,e3,e4)
        d=F.normalize(d,dim=1)
        return self.prototype_similarity(d)

# A3: Without Both (plain UNet = UNet_Vanilla)
# A4: BCE Loss Only
# A5: Without Augmentation

ABLATION_CONFIGS = {
    'Full PSPENet (Proposed)':       {'model': lambda: PSPENet(), 'aug': True,  'loss': 'combined'},
    'w/o Prototype Similarity':     {'model': lambda: PSPENet_NoProto(), 'aug': True,  'loss': 'combined'},
    'w/o Projection Head':          {'model': lambda: PSPENet_NoProj(),  'aug': True,  'loss': 'combined'},
    'w/o Proto + Proj (Plain UNet)':{'model': lambda: UNet_Vanilla(),   'aug': True,  'loss': 'combined'},
    'w/o Dice Loss (BCE Only)':     {'model': lambda: PSPENet(),         'aug': True,  'loss': 'bce'},
    'w/o Augmentation':             {'model': lambda: PSPENet(),         'aug': False, 'loss': 'combined'},
}

ablation_results = {}

for config_name, cfg in ABLATION_CONFIGS.items():
    print(f'\n{"="*60}\n  Ablation: {config_name}\n{"="*60}')
    
    if cfg['aug']:
        abl_train = torch.utils.data.Subset(
            LeafDataset(root, img_size=256, augment=True), train_indices.indices)
    else:
        abl_train = torch.utils.data.Subset(
            LeafDataset(root, img_size=256, augment=False), train_indices.indices)
    
    abl_test = torch.utils.data.Subset(
        LeafDataset(root, img_size=256, augment=False), test_indices.indices)
    abl_train_loader = DataLoader(abl_train, batch_size=8, shuffle=True, num_workers=2)
    abl_test_loader = DataLoader(abl_test, batch_size=8, shuffle=False, num_workers=2)
    
    m = cfg['model']().to(device)
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    
    if cfg['loss'] == 'bce':
        crit = nn.BCELoss()
    else:
        crit = CombinedLoss()
    
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', patience=5, factor=0.5)
    best_iou, best_dice = 0.0, 0.0
    
    for epoch in range(COMP_EPOCHS):
        m.train()
        for imgs, masks in abl_train_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            preds = m(imgs); loss = crit(preds, masks)
            opt.zero_grad(); loss.backward(); opt.step()
        
        met = evaluate_model(m, abl_test_loader)
        sched.step(1 - met['IoU'])
        if met['IoU'] > best_iou:
            best_iou = met['IoU']; best_dice = met['Dice']
            best_met = met.copy()
        
        if (epoch+1) % 10 == 0:
            print(f'    Epoch [{epoch+1}/{COMP_EPOCHS}] IoU: {met["IoU"]:.4f} | Best: {best_iou:.4f}')
    
    ablation_results[config_name] = best_met
    print(f'  ✅ {config_name} → IoU: {best_iou:.4f} | Dice: {best_dice:.4f}')

# Print Ablation Table
abl_df = pd.DataFrame([
    {'Method': k, 'Dataset': 'ATLDSD', 'DICE': f"{v['Dice']:.4f}", 'IOU': f"{v['IoU']:.4f}",
     'Acc': f"{v['Acc']:.4f}", 'F1': f"{v['F1']:.4f}"}
    for k,v in ablation_results.items()
])
print("\n🔬 ABLATION STUDY TABLE")
print(abl_df.to_string(index=False))

In [ ]:
full_dataset_noaug = LeafDataset(root, img_size=256, augment=False)
full_dataset_aug   = LeafDataset(root, img_size=256, augment=True)

K_VALUES = [5]
kfold_results = {}

for k in K_VALUES:
    print(f'\n{"="*60}\n  {k}-Fold Cross Validation\n{"="*60}')
    kf = KFold(n_splits=k, shuffle=True, random_state=42)
    fold_ious, fold_dices = [], []
    
    for fold, (tr_idx, te_idx) in enumerate(kf.split(range(len(full_dataset_noaug)))):
        print(f'  Fold {fold+1}/{k}', end=' → ')
        
        fold_train = torch.utils.data.Subset(full_dataset_aug, tr_idx)
        fold_test  = torch.utils.data.Subset(full_dataset_noaug, te_idx)
        fold_tr_loader = DataLoader(fold_train, batch_size=8, shuffle=True, num_workers=2)
        fold_te_loader = DataLoader(fold_test, batch_size=8, shuffle=False, num_workers=2)
        
        m = PSPENet().to(device)
        opt = torch.optim.Adam(m.parameters(), lr=1e-3)
        crit = CombinedLoss()
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', patience=5, factor=0.5)
        
        best_fold_iou, best_fold_dice = 0.0, 0.0
        for epoch in range(30):  # 30 epochs per fold to save time
            m.train()
            for imgs, masks in fold_tr_loader:
                imgs, masks = imgs.to(device), masks.to(device)
                preds = m(imgs); loss = crit(preds, masks)
                opt.zero_grad(); loss.backward(); opt.step()
            
            met = evaluate_model(m, fold_te_loader)
            sched.step(1 - met['IoU'])
            if met['IoU'] > best_fold_iou:
                best_fold_iou = met['IoU']; best_fold_dice = met['Dice']
        
        fold_ious.append(best_fold_iou)
        fold_dices.append(best_fold_dice)
        print(f'IoU: {best_fold_iou:.4f} | Dice: {best_fold_dice:.4f}')
    
    kfold_results[k] = {'ious': fold_ious, 'dices': fold_dices,
                          'mean_iou': np.mean(fold_ious), 'std_iou': np.std(fold_ious),
                          'mean_dice': np.mean(fold_dices), 'std_dice': np.std(fold_dices)}
    print(f'  Summary → IoU: {np.mean(fold_ious):.4f}±{np.std(fold_ious):.4f} | '
          f'Dice: {np.mean(fold_dices):.4f}±{np.std(fold_dices):.4f}')

# Print K-Fold Table
kf_rows = []
for k, res in kfold_results.items():
    for i in range(k):
        kf_rows.append({'Method':'PSPENet','Dataset':'ATLDSD',
                         'K-Value':f'{k}-Fold (Fold {i+1})',
                         'DICE':f"{res['dices'][i]:.4f}",'IOU':f"{res['ious'][i]:.4f}"})
    kf_rows.append({'Method':'PSPENet','Dataset':'ATLDSD',
                     'K-Value':f'{k}-Fold (Mean±Std)',
                     'DICE':f"{res['mean_dice']:.4f}±{res['std_dice']:.4f}",
                     'IOU':f"{res['mean_iou']:.4f}±{res['std_iou']:.4f}"})

kfold_df = pd.DataFrame(kf_rows)
print("\n🔄 K-FOLD VALIDATION TABLE")
print(kfold_df.to_string(index=False))

In [ ]:
# Load both models
unet_baseline = trained_models['UNet']
unet_baseline.eval()
model.load_state_dict(torch.load('best_model.pth'))
model.eval()

base_ious, base_dices = [], []
prop_ious, prop_dices = [], []
all_base_preds, all_prop_preds, all_targets = [], [], []

with torch.no_grad():
    for imgs, masks in test_loader:
        imgs, masks = imgs.to(device), masks.to(device)
        bp = unet_baseline(imgs)
        pp = model(imgs)
        for j in range(imgs.size(0)):
            base_ious.append(iou_score(bp[j:j+1],masks[j:j+1]).item())
            base_dices.append(dice_score(bp[j:j+1],masks[j:j+1]).item())
            prop_ious.append(iou_score(pp[j:j+1],masks[j:j+1]).item())
            prop_dices.append(dice_score(pp[j:j+1],masks[j:j+1]).item())
            all_base_preds.extend((bp[j]>0.5).cpu().numpy().flatten().tolist())
            all_prop_preds.extend((pp[j]>0.5).cpu().numpy().flatten().tolist())
            all_targets.extend(masks[j].cpu().numpy().flatten().tolist())

base_ious=np.array(base_ious); base_dices=np.array(base_dices)
prop_ious=np.array(prop_ious); prop_dices=np.array(prop_dices)

# 1. Confidence Intervals
def ci(data, conf=0.95):
    n=len(data); m=np.mean(data); se=sem(data)
    h=se*scipy_stats.t.ppf((1+conf)/2,n-1)
    return m, m-h, m+h

b_iou_m,b_iou_lo,b_iou_hi = ci(base_ious)
p_iou_m,p_iou_lo,p_iou_hi = ci(prop_ious)
b_dice_m,b_dice_lo,b_dice_hi = ci(base_dices)
p_dice_m,p_dice_lo,p_dice_hi = ci(prop_dices)

# 2. Paired t-test
t_stat, p_ttest = ttest_rel(prop_ious, base_ious)

# 3. Wilcoxon
w_stat, p_wilcox = wilcoxon(prop_ious, base_ious)

# 4. Cohen's Kappa
base_kappa = cohen_kappa_score(all_targets, all_base_preds)
prop_kappa = cohen_kappa_score(all_targets, all_prop_preds)

# Print
print("\n" + "="*90)
print("  STATISTICAL ANALYSIS")
print("="*90)
print(f"  UNet Baseline  → IoU: {b_iou_m:.4f} [{b_iou_lo:.4f}-{b_iou_hi:.4f}] | κ={base_kappa:.4f}")
print(f"  PSPENet Proposed → IoU: {p_iou_m:.4f} [{p_iou_lo:.4f}-{p_iou_hi:.4f}] | κ={prop_kappa:.4f}")
print(f"  Paired t-test  → t={t_stat:.4f}, p={p_ttest:.6f} {'✅' if p_ttest<0.05 else '❌'}")
print(f"  Wilcoxon       → W={w_stat:.4f}, p={p_wilcox:.6f} {'✅' if p_wilcox<0.05 else '❌'}")

stat_df = pd.DataFrame({
    'Method': ['UNet (Baseline)', 'PSPENet (Proposed)'],
    'Dataset': ['ATLDSD','ATLDSD'],
    '95% CI (IoU)': [f'{b_iou_m:.4f} [{b_iou_lo:.4f}-{b_iou_hi:.4f}]',
                     f'{p_iou_m:.4f} [{p_iou_lo:.4f}-{p_iou_hi:.4f}]'],
    'Wilcoxon p': ['—', f'{p_wilcox:.6f}'],
    'Paired t p': ['—', f'{p_ttest:.6f}'],
    'P-Value': ['—', f'{"<0.05 ✅" if p_ttest<0.05 else "≥0.05 ❌"}'],
    "Cohen's κ": [f'{base_kappa:.4f}', f'{prop_kappa:.4f}'],
})
print("\n📈 STATISTICAL ANALYSIS TABLE")
print(stat_df.to_string(index=False))

In [ ]:
comp_results = {}
dummy = torch.randn(1,3,256,256).to(device)

all_model_objs = {**{k: v for k,v in ALL_MODELS.items()}, 'PSPENet (Proposed)': lambda: PSPENet()}

for name, model_fn in all_model_objs.items():
    m = model_fn().to(device)
    
    # Try load best weights
    try:
        fname = name.replace(' ','_').replace('(','').replace(')','')
        m.load_state_dict(torch.load(f'{fname}.pth'))
    except:
        if name == 'PSPENet (Proposed)':
            m.load_state_dict(torch.load('best_model.pth'))
    m.eval()
    
    # Parameters
    params = sum(p.numel() for p in m.parameters()) / 1e6
    
    # FLOPs
    try:
        flops, _ = profile(m, inputs=(dummy,), verbose=False)
        flops_g = flops / 1e9
    except: flops_g = 0.0
    
    # Inference Time
    times = []
    with torch.no_grad():
        for _ in range(3):
            t0=time.time()
            for imgs, masks in test_loader:
                _ = m(imgs.to(device))
            times.append(time.time()-t0)
    infer_per_img = np.mean(times)/len(test_loader.dataset)
    
    # Memory
    torch.cuda.reset_peak_memory_stats(); torch.cuda.empty_cache()
    with torch.no_grad(): _ = m(dummy)
    memory = torch.cuda.max_memory_allocated()/(1024**2)
    
    comp_results[name] = {
        'Train_Time': all_results.get(name,{}).get('Train_Time',
                      all_results.get('PSPENet (Proposed)',{}).get('Train_Time',0)),
        'Infer': infer_per_img, 'Memory': memory,
        'FLOPs': flops_g, 'Params': params}

comp_df = pd.DataFrame([
    {'Method': k, 'Dataset': 'ATLDSD',
     'Training Time (Sec)': f"{v['Train_Time']:.2f}",
     'Memory Usage (MB)': f"{v['Memory']:.2f}",
     'Inference Time (Sec)': f"{v['Infer']:.4f}",
     'FLOPs (G)': f"{v['FLOPs']:.2f}",
     'Model Parameters (M)': f"{v['Params']:.2f}"}
    for k,v in comp_results.items()
])
print("\n💻 COMPUTATIONAL COST TABLE")
print(comp_df.to_string(index=False))

In [ ]:
hyper_df = pd.DataFrame({
    'Parameter': ['Loss Function','Epochs','Batch Size','Learning Rate',
                  'Optimizer','Activation Function','Train:Test Ratio',
                  'Dropout','Scheduler','Image Size','Embedding Dim','Prototypes'],
    'Value': ['BCELoss + DiceLoss','100','8','1e-3',
              'Adam','Sigmoid (output), ReLU (hidden)','80:20',
              'None (BatchNorm used)','ReduceLROnPlateau (patience=5, factor=0.5)',
              '256×256','64','2 (Background + Lesion)']
})
print("\n⚙️ HYPERPARAMETERS TABLE")
print(hyper_df.to_string(index=False))

In [ ]:
from torchsummary import summary
print("\n📋 MODEL SUMMARY: PSPENet (Proposed)")
summary(PSPENet().to(device), (3,256,256))

In [ ]:
def block_summary(model, input_size=(3, 256, 256)):
    device = next(model.parameters()).device
    x = torch.randn(1, *input_size).to(device)
    
    print("=" * 75)
    print(f"  PSPENet - Block-Wise Model Summary")
    print(f"  Input: {input_size[1]}×{input_size[2]}×{input_size[0]}")
    print("=" * 75)
    print(f"{'Block':<35} {'Output Shape':<22} {'Parameters':>12}")
    print("-" * 75)
    
    total_params = 0
    
    model.eval()
    with torch.no_grad():
        e1 = model.encoder.enc1(x)
        e2 = model.encoder.enc2(model.encoder.pool(e1))
        e3 = model.encoder.enc3(model.encoder.pool(e2))
        e4 = model.encoder.enc4(model.encoder.pool(e3))
        bn = model.encoder.bottleneck(model.encoder.pool(e4))
        
        d4 = model.decoder.dec4(torch.cat([model.decoder.up4(bn), e4], dim=1))
        d3 = model.decoder.dec3(torch.cat([model.decoder.up3(d4), e3], dim=1))
        d2 = model.decoder.dec2(torch.cat([model.decoder.up2(d3), e2], dim=1))
        d1 = model.decoder.dec1(torch.cat([model.decoder.up1(d2), e1], dim=1))
        
        z = model.projection(d1)
        out = model.prototype_similarity(z)
        
        # (name, module_for_params, shape_tensor_or_None)
        blocks = [
            # Encoder
            ("ENCODER", None, None),
            ("  Conv Block 1 (3→64)",    model.encoder.enc1,       e1),
            ("  Conv Block 2 (64→128)",  model.encoder.enc2,       e2),
            ("  Conv Block 3 (128→256)", model.encoder.enc3,       e3),
            ("  Conv Block 4 (256→512)", model.encoder.enc4,       e4),
            # Bottleneck
            ("BOTTLENECK", None, None),
            ("  Conv Block (512→1024)",  model.encoder.bottleneck, bn),
            # Decoder
            ("DECODER", None, None),
            ("  Up4+Conv (1024→512)",    model.decoder.dec4,       d4),
            ("  Up3+Conv (512→256)",     model.decoder.dec3,       d3),
            ("  Up2+Conv (256→128)",     model.decoder.dec2,       d2),
            ("  Up1+Conv (128→64)",      model.decoder.dec1,       d1),
            # Novel
            ("PROJECTION HEAD (Novel)", None, None),
            ("  1×1 Conv+L2Norm (64→64)",model.projection,        z),
            ("PROTOTYPE SIMILARITY (Novel)", None, None),
            ("  2 Prototypes+CosSim",    model.prototype_similarity, out),
        ]
        
        for name, module, tensor in blocks:
            if module is None:
                # Section header
                print(f"\n  {'─'*20} {name} {'─'*20}")
            else:
                n_params = sum(pp.numel() for pp in module.parameters())
                total_params += n_params
                shape_str = f"({tensor.shape[1]}, {tensor.shape[2]}, {tensor.shape[3]})"
                print(f"  {name:<34} {shape_str:<22} {n_params:>10,}")
    
    total_all = sum(pp.numel() for pp in model.parameters())
    trainable = sum(pp.numel() for pp in model.parameters() if pp.requires_grad)
    
    print("\n" + "=" * 75)
    print(f"  Total Parameters        : {total_all:>15,}")
    print(f"  Trainable Parameters    : {trainable:>15,}")
    print(f"  Non-Trainable Parameters: {total_all - trainable:>15,}")
    print(f"  Model Size (MB)         : {total_all * 4 / 1e6:>15.2f}")
    print("=" * 75)

block_summary(PSPENet().to(device))

In [ ]:
from scipy.ndimage import binary_erosion

unet_baseline = trained_models['UNet'].eval()
model.load_state_dict(torch.load('best_model.pth'))
model.eval()

mean_t = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std_t  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

base_ds = test_ds.dataset

# Find samples WITH lesions
lesion_indices = []
for idx in test_ds.indices:
    mask_path = base_ds.masks[idx]
    rgb_mask = np.array(Image.open(mask_path).convert('RGB'))
    is_bg = np.all(rgb_mask == [0, 0, 0], axis=2)
    is_leaf = np.all(rgb_mask == [128, 0, 0], axis=2)
    is_lesion = ~is_bg & ~is_leaf
    if is_lesion.sum() > 500:
        lesion_indices.append(idx)

print(f"Samples with visible lesions: {len(lesion_indices)}")

# ============ MAIN PLOT (5 columns, no difference map, no blue) ============
fig, axes = plt.subplots(5, 5, figsize=(25, 25))
cols = ['Input Image',
        'Ground Truth\n(Black=BG, Red=Leaf,\nColor=Lesion)',
        'GT Overlay\n(Red=Leaf, Yellow=Lesion)',
        'PSPENet Segmentation (Proposed)',
        'UNet Segmentation\n']

for c, t in enumerate(cols):
    axes[0, c].set_title(t, fontsize=18, fontweight='bold')

with torch.no_grad():
    sample_indices = lesion_indices[:5] if len(lesion_indices) >= 5 else list(test_ds.indices)[:5]
    
    for i, idx in enumerate(sample_indices):
        img_t, mask_t = base_ds[idx]
        img_t_dev = img_t.unsqueeze(0).to(device)
        mask_t_dev = mask_t.unsqueeze(0).to(device)
        
        unet_pred = unet_baseline(img_t_dev)
        prop_pred = model(img_t_dev)
        
        # Denormalize
        img_show = (img_t * std_t + mean_t).permute(1, 2, 0).clamp(0, 1).numpy()
        
        # Load RGB mask
        mask_path = base_ds.masks[idx]
        rgb_mask = np.array(Image.open(mask_path).convert('RGB'))
        rgb_mask = np.array(Image.fromarray(rgb_mask).resize((256, 256), Image.NEAREST))
        
        # Separate regions
        is_bg = np.all(rgb_mask == [0, 0, 0], axis=2)
        is_leaf = np.all(rgb_mask == [128, 0, 0], axis=2)
        is_lesion = ~is_bg & ~is_leaf
        
        gt_binary = mask_t[0].cpu().numpy()
        
        # Predictions
        unet_bin = (unet_pred[0, 0].cpu() > 0.5).float().numpy()
        prop_bin = (prop_pred[0, 0].cpu() > 0.5).float().numpy()
        
        u_iou = iou_score(unet_pred, mask_t_dev).item()
        p_iou = iou_score(prop_pred, mask_t_dev).item()
        
        # ---- Col 0: Input Image ----
        axes[i, 0].imshow(img_show)
        axes[i, 0].axis('off')
        
        # ---- Col 1: RGB Mask ----
        display_mask = np.ones_like(img_show) * 0.2
        display_mask[is_leaf] = [0.8, 0.1, 0.1]
        display_mask[is_lesion] = [1.0, 0.9, 0.0]
        axes[i, 1].imshow(display_mask)
        axes[i, 1].axis('off')
        axes[i, 1].set_xlabel(f'Leaf: {is_leaf.mean()*100:.1f}% | Lesion: {is_lesion.mean()*100:.1f}%', 
                              fontsize=9)
        
        # ---- Col 2: GT Overlay on image ----
        gt_overlay = img_show.copy()
        gt_overlay[is_leaf] = gt_overlay[is_leaf] * 0.5 + np.array([0.8, 0.1, 0.1]) * 0.5
        gt_overlay[is_lesion] = gt_overlay[is_lesion] * 0.3 + np.array([1.0, 0.9, 0.0]) * 0.7
        # Lesion boundary
        if is_lesion.any():
            lesion_interior = binary_erosion(is_lesion, iterations=2)
            lesion_boundary = is_lesion & ~lesion_interior
            gt_overlay[lesion_boundary] = [1.0, 1.0, 0.0]
        axes[i, 2].imshow(gt_overlay)
        axes[i, 2].axis('off')
        
        # ---- Col 3: UNet prediction ----
        unet_vis = img_show.copy()
        tp_leaf = (unet_bin == 1) & is_leaf
        tp_lesion = (unet_bin == 1) & is_lesion
        fn = (unet_bin == 0) & (gt_binary == 1)
        
        unet_vis[tp_leaf] = unet_vis[tp_leaf] * 0.4 + np.array([0, 0.7, 0]) * 0.6
        unet_vis[tp_lesion] = unet_vis[tp_lesion] * 0.3 + np.array([1.0, 0.9, 0.0]) * 0.7
        unet_vis[fn] = unet_vis[fn] * 0.3 + np.array([0.8, 0, 0]) * 0.7
        # Leaf boundary
        unet_interior = binary_erosion(unet_bin, iterations=2)
        unet_boundary = unet_bin.astype(bool) & ~unet_interior
        unet_vis[unet_boundary] = [0, 1, 0]
        # Lesion boundary on prediction
        if tp_lesion.any():
            lesion_pred_interior = binary_erosion(tp_lesion, iterations=2)
            lesion_pred_boundary = tp_lesion & ~lesion_pred_interior
            unet_vis[lesion_pred_boundary] = [1.0, 1.0, 0.0]
        axes[i, 3].imshow(unet_vis)
        axes[i, 3].axis('off')
        axes[i, 3].set_xlabel(f'IoU: {u_iou:.4f}', fontsize=11)
        
        # ---- Col 4: PSPENet prediction ----
        prop_vis = img_show.copy()
        tp_leaf_p = (prop_bin == 1) & is_leaf
        tp_lesion_p = (prop_bin == 1) & is_lesion
        fn_p = (prop_bin == 0) & (gt_binary == 1)
        
        prop_vis[tp_leaf_p] = prop_vis[tp_leaf_p] * 0.4 + np.array([0, 0.7, 0]) * 0.6
        prop_vis[tp_lesion_p] = prop_vis[tp_lesion_p] * 0.3 + np.array([1.0, 0.9, 0.0]) * 0.7
        prop_vis[fn_p] = prop_vis[fn_p] * 0.3 + np.array([0.8, 0, 0]) * 0.7
        # Leaf boundary
        prop_interior = binary_erosion(prop_bin, iterations=2)
        prop_boundary = prop_bin.astype(bool) & ~prop_interior
        prop_vis[prop_boundary] = [0, 1, 0]
        # Lesion boundary on prediction
        if tp_lesion_p.any():
            lesion_prop_interior = binary_erosion(tp_lesion_p, iterations=2)
            lesion_prop_boundary = tp_lesion_p & ~lesion_prop_interior
            prop_vis[lesion_prop_boundary] = [1.0, 1.0, 0.0]
        axes[i, 4].imshow(prop_vis)
        axes[i, 4].axis('off')
        axes[i, 4].set_xlabel(f'IoU: {p_iou:.4f}', fontsize=11, fontweight='bold', color='green')

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='green', label='Correct Leaf'),
    Patch(facecolor='yellow', edgecolor='black', label='Correct Lesion'),
    Patch(facecolor='red', label='Missed (FN)'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=3, fontsize=12,
           bbox_to_anchor=(0.5, -0.01), frameon=True, edgecolor='black')

plt.tight_layout()
save_fig(fig, 'comparison_leaf_segmentation')
plt.show()

In [ ]:
unet_baseline = trained_models['UNet'].eval()
model.load_state_dict(torch.load('best_model.pth'))
model.eval()

mean_t = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std_t  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

base_ds = test_ds.dataset  # LeafDataset

fig, axes = plt.subplots(5, 6, figsize=(30, 25))
cols = ['Input Image', 
        'GT: Leaf Region\n(Red)', 
        'GT: Lesion Region\n(Disease Color)',
        'GT: Combined Mask',
        'UNet Prediction', 
        'PSPENet Prediction\n(Proposed)']

for c, t in enumerate(cols):
    axes[0, c].set_title(t, fontsize=12, fontweight='bold')

with torch.no_grad():
    test_indices_list = list(test_ds.indices)[:5]
    
    for i, idx in enumerate(test_indices_list):
        img_t, mask_t = base_ds[idx]
        img_t_dev = img_t.unsqueeze(0).to(device)
        mask_t_dev = mask_t.unsqueeze(0).to(device)
        
        unet_pred = unet_baseline(img_t_dev)
        prop_pred = model(img_t_dev)
        
        # Denormalize image
        img_show = (img_t * std_t + mean_t).permute(1, 2, 0).clamp(0, 1).numpy()
        
        # Load RGB mask
        mask_path = base_ds.masks[idx]
        rgb_mask = np.array(Image.open(mask_path).convert('RGB'))
        rgb_mask = np.array(Image.fromarray(rgb_mask).resize((256, 256), Image.NEAREST))
        
        # Separate leaf vs lesion
        is_bg = np.all(rgb_mask == [0, 0, 0], axis=2)
        is_leaf = np.all(rgb_mask == [128, 0, 0], axis=2)
        is_lesion = ~is_bg & ~is_leaf  # everything else is disease
        
        # Create colored visualization for leaf (red overlay)
        leaf_overlay = img_show.copy()
        leaf_overlay[is_leaf] = leaf_overlay[is_leaf] * 0.4 + np.array([1, 0, 0]) * 0.6
        
        # Create colored visualization for lesion (yellow overlay)
        lesion_overlay = img_show.copy()
        lesion_overlay[is_lesion] = lesion_overlay[is_lesion] * 0.3 + np.array([1, 1, 0]) * 0.7
        
        # Combined: leaf=red, lesion=yellow
        combined_overlay = img_show.copy()
        combined_overlay[is_leaf] = combined_overlay[is_leaf] * 0.5 + np.array([1, 0, 0]) * 0.5
        combined_overlay[is_lesion] = combined_overlay[is_lesion] * 0.3 + np.array([1, 1, 0]) * 0.7
        
        # Predictions (binary = leaf+lesion)
        unet_bin = (unet_pred[0, 0].cpu() > 0.5).float().numpy()
        prop_bin = (prop_pred[0, 0].cpu() > 0.5).float().numpy()
        
        # UNet overlay: predicted region in green
        overlay_u = img_show.copy()
        overlay_u[unet_bin == 1] = overlay_u[unet_bin == 1] * 0.4 + np.array([0, 1, 0]) * 0.6
        
        # PSPENet overlay: predicted region in green
        overlay_p = img_show.copy()
        overlay_p[prop_bin == 1] = overlay_p[prop_bin == 1] * 0.4 + np.array([0, 1, 0]) * 0.6
        
        # IoU scores
        u_iou = iou_score(unet_pred, mask_t_dev).item()
        p_iou = iou_score(prop_pred, mask_t_dev).item()
        
        # Plot
        axes[i, 0].imshow(img_show); axes[i, 0].axis('off')
        
        axes[i, 1].imshow(leaf_overlay); axes[i, 1].axis('off')
        axes[i, 1].set_xlabel(f'Leaf: {is_leaf.mean()*100:.1f}%', fontsize=10)
        
        axes[i, 2].imshow(lesion_overlay); axes[i, 2].axis('off')
        axes[i, 2].set_xlabel(f'Lesion: {is_lesion.mean()*100:.1f}%', fontsize=10, 
                              color='darkorange', fontweight='bold')
        
        axes[i, 3].imshow(combined_overlay); axes[i, 3].axis('off')
        axes[i, 3].set_xlabel(f'Red=Leaf, Yellow=Lesion', fontsize=9)
        
        axes[i, 4].imshow(overlay_u); axes[i, 4].axis('off')
        axes[i, 4].set_xlabel(f'IoU: {u_iou:.4f}', fontsize=11)
        
        axes[i, 5].imshow(overlay_p); axes[i, 5].axis('off')
        axes[i, 5].set_xlabel(f'IoU: {p_iou:.4f}', fontsize=11, 
                              fontweight='bold', color='green')

plt.suptitle('Apple Leaf Disease Segmentation: UNet vs PSPENet (Proposed)', 
             fontsize=16, fontweight='bold')
plt.tight_layout()
save_fig(fig, 'comparison_results_detailed')
plt.show()

In [ ]:
class LeafDataset(Dataset):
    def __init__(self, root, img_size=256, augment=False):
        self.images, self.masks = [], []
        self.augment = augment
        self.img_size = img_size
        self.normalize = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        for cls in os.listdir(root):
            img_dir = os.path.join(root, cls, 'image')
            lbl_dir = os.path.join(root, cls, 'label')
            if not os.path.isdir(img_dir): continue
            for f in sorted(os.listdir(img_dir)):
                lbl_f = os.path.splitext(f)[0] + '.png'
                lbl_path = os.path.join(lbl_dir, lbl_f)
                if os.path.exists(lbl_path):
                    self.images.append(os.path.join(img_dir, f))
                    self.masks.append(lbl_path)

    def __len__(self): return len(self.images)

    def __getitem__(self, i):
        img = Image.open(self.images[i]).convert('RGB')
        
        # ✅ CHANGED: Load as RGB to separate leaf vs lesion
        mask_rgb = np.array(Image.open(self.masks[i]).convert('RGB'))
        
        # ✅ Background = [0,0,0], Healthy leaf = [128,0,0]
        # ✅ Lesion = everything ELSE
        is_background = np.all(mask_rgb == [0, 0, 0], axis=2)
        is_healthy_leaf = np.all(mask_rgb == [128, 0, 0], axis=2)
        lesion_mask = (~is_background & ~is_healthy_leaf).astype(np.float32)
        
        mask = Image.fromarray(lesion_mask)

        # Resize both
        img = transforms.functional.resize(img, (self.img_size, self.img_size))
        mask = transforms.functional.resize(mask, (self.img_size, self.img_size), 
                                            interpolation=transforms.InterpolationMode.NEAREST)
        if self.augment:
            if random.random() > 0.5:
                img = transforms.functional.hflip(img)
                mask = transforms.functional.hflip(mask)
            if random.random() > 0.5:
                img = transforms.functional.vflip(img)
                mask = transforms.functional.vflip(mask)
            angle = random.uniform(-30, 30)
            img = transforms.functional.rotate(img, angle)
            mask = transforms.functional.rotate(mask, angle)
            img = transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1)(img)
        
        img = transforms.functional.to_tensor(img)
        img = self.normalize(img)
        mask = transforms.functional.to_tensor(mask)
        mask = (mask > 0.5).float()  # clean interpolation artifacts

        return img, mask

In [ ]:
unet_baseline = trained_models['UNet'].eval()
model.load_state_dict(torch.load('best_model.pth'))
model.eval()

mean_t = torch.tensor([0.485,0.456,0.406]).view(3,1,1)
std_t  = torch.tensor([0.229,0.224,0.225]).view(3,1,1)

# Get underlying dataset to access mask paths
base_ds = test_ds.dataset  # LeafDataset

fig, axes = plt.subplots(5, 4, figsize=(20, 25))
cols = ['Input Image', 'Ground Truth\n(Red=Leaf, Color=Lesion)', 
        'UNet Prediction', 'PSPENet Prediction (Proposed)']
for c, t in enumerate(cols):
    axes[0, c].set_title(t, fontsize=13, fontweight='bold')

with torch.no_grad():
    test_imgs, test_masks = [], []
    test_indices_list = list(test_ds.indices)[:5]
    
    for i, idx in enumerate(test_indices_list):
        img_t, mask_t = base_ds[idx]
        img_t, mask_t = img_t.unsqueeze(0).to(device), mask_t.unsqueeze(0).to(device)
        
        unet_pred = unet_baseline(img_t)
        prop_pred = model(img_t)
        
        # Denormalize image
        img_show = (img_t[0].cpu()*std_t+mean_t).permute(1,2,0).clamp(0,1).numpy()
        
        # Load RGB mask directly
        mask_path = base_ds.masks[idx]
        rgb_mask = np.array(Image.open(mask_path).convert('RGB'))
        rgb_mask = np.array(Image.fromarray(rgb_mask).resize((256,256), Image.NEAREST))
        
        unet_bin = (unet_pred[0,0].cpu()>0.5).float().numpy()
        prop_bin = (prop_pred[0,0].cpu()>0.5).float().numpy()
        
        # Col 0: Input
        axes[i,0].imshow(img_show); axes[i,0].axis('off')
        
        # Col 1: RGB Mask (red=leaf, color=lesion)
        axes[i,1].imshow(rgb_mask); axes[i,1].axis('off')
        
        # Col 2: UNet overlay
        overlay_u = img_show.copy()
        overlay_u[unet_bin==1] = overlay_u[unet_bin==1]*0.4 + np.array([0,1,0])*0.6
        axes[i,2].imshow(overlay_u); axes[i,2].axis('off')
        u_iou = iou_score(unet_pred, mask_t).item()
        axes[i,2].set_xlabel(f'IoU: {u_iou:.4f}', fontsize=11)
        
        # Col 3: Proposed overlay
        overlay_p = img_show.copy()
        overlay_p[prop_bin==1] = overlay_p[prop_bin==1]*0.4 + np.array([0,1,0])*0.6
        axes[i,3].imshow(overlay_p); axes[i,3].axis('off')
        p_iou = iou_score(prop_pred, mask_t).item()
        axes[i,3].set_xlabel(f'IoU: {p_iou:.4f}', fontsize=11, fontweight='bold', color='green')

plt.suptitle('Segmentation Comparison: UNet vs PSPENet (Proposed)', fontsize=16, fontweight='bold')
plt.tight_layout()
save_fig(fig, 'comparison_results')
plt.show()

In [ ]:
# Register hooks
activations = {}
def get_hook(name):
    def hook(model, input, output):
        activations[name] = output.detach()
    return hook

# UNet hooks
h1 = unet_baseline.encoder.enc1.register_forward_hook(get_hook('unet_enc1'))
h2 = unet_baseline.encoder.enc4.register_forward_hook(get_hook('unet_enc4'))

# Proposed hooks
h3 = model.encoder.enc1.register_forward_hook(get_hook('pspeunet_enc1'))
h4 = model.encoder.enc4.register_forward_hook(get_hook('pspeunet_enc4'))

# Forward pass
sample_idx = test_ds.indices[0]
img_t, mask_t = base_ds[sample_idx]
img_t = img_t.unsqueeze(0).to(device)
img_show = (img_t[0].cpu()*std_t+mean_t).permute(1,2,0).clamp(0,1).numpy()

with torch.no_grad():
    _ = unet_baseline(img_t)
    _ = model(img_t)

# Remove hooks
h1.remove(); h2.remove(); h3.remove(); h4.remove()

# Plot
fig, axes = plt.subplots(3, 5, figsize=(25, 15))

layer_names = ['unet_enc1','unet_enc4','pspeunet_enc1','pspeunet_enc4']
titles = ['UNet — Layer 1\n(Shallow Features)',
          'UNet — Layer 4\n(Deep Features)',
          'PSPENet — Layer 1\n(Shallow Features)',
          'PSPENet — Layer 4\n(Deep Features)']

# Row 0: Input + Average activations
axes[0,0].imshow(img_show); axes[0,0].set_title('Input Image', fontsize=22, fontweight='bold'); axes[0,0].axis('off')
for j, (ln, tt) in enumerate(zip(layer_names, titles)):
    feat = activations[ln][0].mean(dim=0).cpu().numpy()  # average across channels
    feat = (feat - feat.min()) / (feat.max() - feat.min() + 1e-8)
    axes[0,j+1].imshow(feat, cmap='jet'); axes[0,j+1].set_title(tt, fontsize=18, fontweight='bold'); axes[0,j+1].axis('off')

# Row 1: Individual channels (first 5 channels of enc1)
axes[1,0].set_ylabel('Channel Samples\n(Layer 1)', fontsize=12, fontweight='bold')
for j in range(5):
    feat_u = activations['unet_enc1'][0,j].cpu().numpy()
    feat_u = (feat_u - feat_u.min())/(feat_u.max()-feat_u.min()+1e-8)
    axes[1,j].imshow(feat_u, cmap='viridis')
    axes[1,j].set_title(f'UNet Ch-{j}' if j<3 else f'PSPENet Ch-{j-3}', fontsize=10)
    axes[1,j].axis('off')

# Overwrite Row 1 properly: 3 UNet channels + 2 PSPENet channels  
for j in range(3):
    feat = activations['unet_enc1'][0,j].cpu().numpy()
    feat = (feat-feat.min())/(feat.max()-feat.min()+1e-8)
    axes[1,j].imshow(feat, cmap='viridis'); axes[1,j].set_title(f'UNet Ch-{j}', fontsize=22); axes[1,j].axis('off')
for j in range(2):
    feat = activations['pspeunet_enc1'][0,j].cpu().numpy()
    feat = (feat-feat.min())/(feat.max()-feat.min()+1e-8)
    axes[1,j+3].imshow(feat, cmap='magma'); axes[1,j+3].set_title(f'PSPENet Ch-{j}', fontsize=22); axes[1,j+3].axis('off')

# Row 2: Individual channels (enc4)
for j in range(3):
    feat = activations['unet_enc4'][0,j].cpu().numpy()
    feat = (feat-feat.min())/(feat.max()-feat.min()+1e-8)
    axes[2,j].imshow(feat, cmap='viridis'); axes[2,j].set_title(f'UNet Deep Ch-{j}', fontsize=22); axes[2,j].axis('off')
for j in range(2):
    feat = activations['pspeunet_enc4'][0,j].cpu().numpy()
    feat = (feat-feat.min())/(feat.max()-feat.min()+1e-8)
    axes[2,j+3].imshow(feat, cmap='magma'); axes[2,j+3].set_title(f'PSPENet Deep Ch-{j}', fontsize=22); axes[2,j+3].axis('off')

# plt.suptitle('Feature Map Visualization: UNet vs PSPENet', fontsize=16, fontweight='bold')
plt.tight_layout()
save_fig(fig, 'feature_maps')
plt.show()

In [ ]:
# Register hooks on decoder output (before final layer)
dec_activations = {}

def dec_hook(name):
    def hook(m, inp, out):
        dec_activations[name] = out.detach()
    return hook

# Hook on decoder output for both models
h_u = unet_baseline.decoder.dec1.register_forward_hook(dec_hook('unet_dec'))
h_p = model.decoder.dec1.register_forward_hook(dec_hook('pspeunet_dec'))

fig, axes = plt.subplots(3, 4, figsize=(25, 20))
col_titles = ['Input Image', 'Ground Truth', 'UNet Heatmap', 'PSPENet Heatmap']

for c, t in enumerate(col_titles):
    axes[0, c].set_title(t, fontsize=18, fontweight='bold')

for row in range(3):
    idx = test_ds.indices[row]
    img_t, mask_t = base_ds[idx]
    img_t_dev = img_t.unsqueeze(0).to(device)
    mask_t_dev = mask_t.unsqueeze(0).to(device)
    
    with torch.no_grad():
        u_pred = unet_baseline(img_t_dev)
        p_pred = model(img_t_dev)
    
    img_show = (img_t*std_t+mean_t).permute(1,2,0).clamp(0,1).numpy()
    
    # RGB mask
    mask_path = base_ds.masks[idx]
    rgb_mask = np.array(Image.open(mask_path).convert('RGB').resize((256,256), Image.NEAREST))
    
    # Heatmaps (average activation from decoder)
    unet_heat = dec_activations['unet_dec'][0].mean(0).cpu().numpy()
    unet_heat = (unet_heat - unet_heat.min()) / (unet_heat.max() - unet_heat.min() + 1e-8)
    
    pspeunet_heat = dec_activations['pspeunet_dec'][0].mean(0).cpu().numpy()
    pspenet_heat = (pspeunet_heat - pspeunet_heat.min()) / (pspeunet_heat.max() - pspeunet_heat.min() + 1e-8)
    
    prop_bin = (p_pred[0,0].cpu() > 0.5).float().numpy()
    
    # Col 0: Input
    axes[row,0].imshow(img_show); axes[row,0].axis('off')
    
    # Col 1: GT RGB mask
    axes[row,1].imshow(rgb_mask); axes[row,1].axis('off')
    
    # Col 2: UNet heatmap overlay
    axes[row,2].imshow(img_show)
    axes[row,2].imshow(unet_heat, cmap='jet', alpha=0.5)
    axes[row,2].axis('off')
    
    # Col 3: PSPENet heatmap overlay
    axes[row,3].imshow(img_show)
    axes[row,3].imshow(pspeunet_heat, cmap='jet', alpha=0.5)
    axes[row,3].axis('off')
    
    # Col 4: PSPENet prediction overlay
    # overlay = img_show.copy()
    # overlay[prop_bin==1] = overlay[prop_bin==1]*0.4 + np.array([0,1,0])*0.6
    # axes[row,4].imshow(overlay); axes[row,4].axis('off')
    # p_iou = iou_score(p_pred, mask_t_dev).item()
    # axes[row,4].set_xlabel(f'IoU: {p_iou:.4f}', fontsize=11, fontweight='bold', color='green')

h_u.remove(); h_p.remove()

# plt.suptitle('Activation Heatmap: UNet vs PSPENet (Proposed)', fontsize=16, fontweight='bold')
plt.tight_layout()
save_fig(fig, 'heatmap_comparison')
plt.show()

In [ ]:
print("\n\n" + "#"*100)
print("#" + " "*35 + "ALL RESULTS SUMMARY" + " "*35 + "#")
print("#"*100)

print("\n📊 1. DIFFERENT METHODS")
print(methods_df.to_string(index=False))

print("\n🔬 2. ABLATION STUDY")
print(abl_df.to_string(index=False))

print("\n🔄 3. K-FOLD VALIDATION")
print(kfold_df.to_string(index=False))

print("\n📈 4. STATISTICAL ANALYSIS")
print(stat_df.to_string(index=False))

print("\n💻 5. COMPUTATIONAL COST")
print(comp_df.to_string(index=False))

print("\n⚙️ 6. HYPERPARAMETERS")
print(hyper_df.to_string(index=False))

In [ ]:
model.load_state_dict(torch.load('best_model.pth'))
model.eval()

r_acc, r_prec, r_rec, r_f1 = 0.0, 0.0, 0.0, 0.0
r_iou, r_dice, r_miou, r_mpa = 0.0, 0.0, 0.0, 0.0
total = 0

with torch.no_grad():
    for imgs, masks in test_loader:
        imgs, masks = imgs.to(device), masks.to(device)
        preds = model(imgs)
        bs = imgs.size(0)
        r_acc  += pixel_accuracy(preds, masks).item() * bs
        r_prec += precision_score(preds, masks).item() * bs
        r_rec  += recall_score(preds, masks).item() * bs
        r_f1   += f1_score(preds, masks).item() * bs
        r_iou  += iou_score(preds, masks).item() * bs
        r_dice += dice_score(preds, masks).item() * bs
        r_miou += mean_iou(preds, masks).item() * bs
        r_mpa  += mean_pixel_accuracy(preds, masks).item() * bs
        total  += bs

print("=" * 55)
print("  BEST MODEL - Final Test Metrics")
print("=" * 55)
print(f"  Pixel Accuracy    : {r_acc/total:.4f}")
print(f"  Precision         : {r_prec/total:.4f}")
print(f"  Recall            : {r_rec/total:.4f}")
print(f"  F1 Score          : {r_f1/total:.4f}")
print(f"  IoU (Foreground)  : {r_iou/total:.4f}")
print(f"  Dice Score        : {r_dice/total:.4f}")
print(f"  mIoU              : {r_miou/total:.4f}")
print(f"  mPA               : {r_mpa/total:.4f}")
print("=" * 55)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(train_losses, label='Train Loss')
axes[0].plot(test_losses, label='Test Loss')
axes[0].set_title('Loss'); axes[0].legend()

axes[1].plot(train_ious, label='Train IoU')
axes[1].plot(test_ious, label='Test IoU')
axes[1].set_title('IoU'); axes[1].legend()

axes[2].plot(train_dices, label='Train Dice')
axes[2].plot(test_dices, label='Test Dice')
axes[2].set_title('Dice Score'); axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
def pixel_accuracy(preds, masks, threshold=0.5):
    preds_bin = (preds > threshold).float()
    correct = (preds_bin == masks).sum()
    return correct / masks.numel()

def precision_score(preds, masks, threshold=0.5, smooth=1e-6):
    preds_bin = (preds > threshold).float()
    tp = (preds_bin * masks).sum()
    fp = (preds_bin * (1 - masks)).sum()
    return (tp + smooth) / (tp + fp + smooth)

def recall_score(preds, masks, threshold=0.5, smooth=1e-6):
    preds_bin = (preds > threshold).float()
    tp = (preds_bin * masks).sum()
    fn = ((1 - preds_bin) * masks).sum()
    return (tp + smooth) / (tp + fn + smooth)

def iou_score(preds, masks, threshold=0.5, smooth=1e-6):
    preds_bin = (preds > threshold).float()
    intersection = (preds_bin * masks).sum()
    union = preds_bin.sum() + masks.sum() - intersection
    return (intersection + smooth) / (union + smooth)

def dice_score(preds, masks, threshold=0.5, smooth=1e-6):
    preds_bin = (preds > threshold).float()
    intersection = (preds_bin * masks).sum()
    return (2. * intersection + smooth) / (preds_bin.sum() + masks.sum() + smooth)

In [ ]:
class DiceBCELoss(nn.Module):
    def __init__(self, dice_weight=0.5, bce_weight=0.5, smooth=1e-6):
        super().__init__()
        self.dice_weight = dice_weight
        self.bce_weight = bce_weight
        self.smooth = smooth
        self.bce = nn.BCELoss()

    def forward(self, preds, targets):
        bce_loss = self.bce(preds, targets)
        
        # ✅ Fixed
        preds_flat = preds.reshape(-1)
        targets_flat = targets.reshape(-1)
        intersection = (preds_flat * targets_flat).sum()
        dice = (2. * intersection + self.smooth) / (preds_flat.sum() + targets_flat.sum() + self.smooth)
        dice_loss = 1 - dice
        
        return self.bce_weight * bce_loss + self.dice_weight * dice_loss


class FocalDiceLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0, dice_weight=0.5, focal_weight=0.5, smooth=1e-6):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.dice_weight = dice_weight
        self.focal_weight = focal_weight
        self.smooth = smooth

    def forward(self, preds, targets):
        # Focal Loss
        bce = -(targets * torch.log(preds + 1e-7) + (1 - targets) * torch.log(1 - preds + 1e-7))
        pt = targets * preds + (1 - targets) * (1 - preds)
        alpha_t = targets * self.alpha + (1 - targets) * (1 - self.alpha)
        focal_loss = (alpha_t * (1 - pt) ** self.gamma * bce).mean()
        
        # Dice Loss — ✅ Fixed: .reshape instead of .view
        preds_flat = preds.reshape(-1)
        targets_flat = targets.reshape(-1)
        intersection = (preds_flat * targets_flat).sum()
        dice = (2. * intersection + self.smooth) / (preds_flat.sum() + targets_flat.sum() + self.smooth)
        dice_loss = 1 - dice
        
        return self.focal_weight * focal_loss + self.dice_weight * dice_loss

In [ ]:
# Reinitialize everything
model = PSPENet().to(device)  # or your model class

# ✅ Use combined loss instead of plain BCELoss
criterion = FocalDiceLoss(alpha=0.75, gamma=2.0, dice_weight=0.5, focal_weight=0.5)
# OR simpler option:
# criterion = DiceBCELoss(dice_weight=0.5, bce_weight=0.5)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

# ============ TRAINING LOOP ============
num_epochs = 50
best_test_iou = 0.0

train_losses, test_losses = [], []
train_ious, test_ious = [], []
train_dices, test_dices = [], []
train_accs, test_accs = [], []
train_precisions, test_precisions = [], []
train_recalls, test_recalls = [], []

for epoch in range(num_epochs):
    # ============ TRAINING ============
    model.train()
    running_loss = 0.0
    running_iou, running_dice = 0.0, 0.0
    running_acc, running_prec, running_rec = 0.0, 0.0, 0.0
    total_train = 0

    for imgs, masks in train_loader:
        imgs, masks = imgs.to(device), masks.to(device)
        preds = model(imgs)
        loss = criterion(preds, masks)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        batch_size = imgs.size(0)
        running_loss  += loss.item() * batch_size
        running_iou   += iou_score(preds, masks).item() * batch_size
        running_dice  += dice_score(preds, masks).item() * batch_size
        running_acc   += pixel_accuracy(preds, masks).item() * batch_size
        running_prec  += precision_score(preds, masks).item() * batch_size
        running_rec   += recall_score(preds, masks).item() * batch_size
        total_train   += batch_size

    train_loss = running_loss / total_train
    train_iou  = running_iou / total_train
    train_dice = running_dice / total_train
    train_acc  = running_acc / total_train
    train_prec = running_prec / total_train
    train_rec  = running_rec / total_train

    train_losses.append(train_loss)
    train_ious.append(train_iou)
    train_dices.append(train_dice)
    train_accs.append(train_acc)
    train_precisions.append(train_prec)
    train_recalls.append(train_rec)

    # ============ TESTING ============
    model.eval()
    running_loss = 0.0
    running_iou, running_dice = 0.0, 0.0
    running_acc, running_prec, running_rec = 0.0, 0.0, 0.0
    total_test = 0

    with torch.no_grad():
        for imgs, masks in test_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            preds = model(imgs)
            loss = criterion(preds, masks)

            batch_size = imgs.size(0)
            running_loss  += loss.item() * batch_size
            running_iou   += iou_score(preds, masks).item() * batch_size
            running_dice  += dice_score(preds, masks).item() * batch_size
            running_acc   += pixel_accuracy(preds, masks).item() * batch_size
            running_prec  += precision_score(preds, masks).item() * batch_size
            running_rec   += recall_score(preds, masks).item() * batch_size
            total_test    += batch_size

    test_loss = running_loss / total_test
    test_iou  = running_iou / total_test
    test_dice = running_dice / total_test
    test_acc  = running_acc / total_test
    test_prec = running_prec / total_test
    test_rec  = running_rec / total_test

    test_losses.append(test_loss)
    test_ious.append(test_iou)
    test_dices.append(test_dice)
    test_accs.append(test_acc)
    test_precisions.append(test_prec)
    test_recalls.append(test_rec)

    scheduler.step(test_loss)

    if test_iou > best_test_iou:
        best_test_iou = test_iou
        torch.save(model.state_dict(), 'best_model.pth')
        print(f'  ✅ Best model saved (IoU: {best_test_iou:.4f})')

    print(f'Epoch [{epoch+1}/{num_epochs}]')
    print(f'  Train → Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | IoU: {train_iou:.4f} | '
          f'Dice/F1: {train_dice:.4f} | Prec: {train_prec:.4f} | Rec: {train_rec:.4f}')
    print(f'  Test  → Loss: {test_loss:.4f} | Acc: {test_acc:.4f} | IoU: {test_iou:.4f} | '
          f'Dice/F1: {test_dice:.4f} | Prec: {test_prec:.4f} | Rec: {test_rec:.4f}')
    print('-' * 90)

In [ ]:
# Load best model and evaluate
model.load_state_dict(torch.load('best_model.pth'))
model.eval()

running_iou, running_dice, running_acc = 0.0, 0.0, 0.0
running_prec, running_rec = 0.0, 0.0
total = 0

with torch.no_grad():
    for imgs, masks in test_loader:
        imgs, masks = imgs.to(device), masks.to(device)
        preds = model(imgs)
        bs = imgs.size(0)
        running_acc  += pixel_accuracy(preds, masks).item() * bs
        running_iou  += iou_score(preds, masks).item() * bs
        running_dice += dice_score(preds, masks).item() * bs
        running_prec += precision_score(preds, masks).item() * bs
        running_rec  += recall_score(preds, masks).item() * bs
        total += bs

print("=" * 50)
print("  BEST MODEL - Final Test Metrics")
print("=" * 50)
print(f"  Pixel Accuracy : {running_acc/total:.4f}")
print(f"  IoU            : {running_iou/total:.4f}")
print(f"  Dice (= F1)    : {running_dice/total:.4f}")
print(f"  Precision      : {running_prec/total:.4f}")
print(f"  Recall         : {running_rec/total:.4f}")
print("=" * 50)

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, num_epochs + 1)

fig, axes = plt.subplots(2, 3, figsize=(20, 10))

# Loss
axes[0, 0].plot(epochs, train_losses, 'b-', label='Train')
axes[0, 0].plot(epochs, test_losses, 'r-', label='Test')
axes[0, 0].set_title('Loss', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].legend()
axes[0, 0].grid(True)

# Pixel Accuracy
axes[0, 1].plot(epochs, train_accs, 'b-', label='Train')
axes[0, 1].plot(epochs, test_accs, 'r-', label='Test')
axes[0, 1].set_title('Pixel Accuracy', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].legend()
axes[0, 1].grid(True)

# IoU
axes[0, 2].plot(epochs, train_ious, 'b-', label='Train')
axes[0, 2].plot(epochs, test_ious, 'r-', label='Test')
axes[0, 2].set_title('IoU', fontsize=14, fontweight='bold')
axes[0, 2].set_xlabel('Epoch')
axes[0, 2].legend()
axes[0, 2].grid(True)

# Dice / F1
axes[1, 0].plot(epochs, train_dices, 'b-', label='Train')
axes[1, 0].plot(epochs, test_dices, 'r-', label='Test')
axes[1, 0].set_title('Dice / F1 Score', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].legend()
axes[1, 0].grid(True)

# Precision
axes[1, 1].plot(epochs, train_precisions, 'b-', label='Train')
axes[1, 1].plot(epochs, test_precisions, 'r-', label='Test')
axes[1, 1].set_title('Precision', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].legend()
axes[1, 1].grid(True)

# Recall
axes[1, 2].plot(epochs, train_recalls, 'b-', label='Train')
axes[1, 2].plot(epochs, test_recalls, 'r-', label='Test')
axes[1, 2].set_title('Recall', fontsize=14, fontweight='bold')
axes[1, 2].set_xlabel('Epoch')
axes[1, 2].legend()
axes[1, 2].grid(True)

plt.suptitle('Apple Leaf Disease Segmentation - Training Metrics', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('training_metrics.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({
    'font.size': 16,
    'font.weight': 'bold',
    'axes.labelweight': 'bold',
    'axes.titleweight': 'bold',
})

epochs = range(1, num_epochs + 1)

def style_and_save(ax, title, filename):
    ax.set_title(title, fontsize=20, fontweight='bold', pad=12)
    ax.set_xlabel('Epoch', fontsize=18, fontweight='bold')
    ax.set_ylabel(title, fontsize=18, fontweight='bold')
    ax.legend(fontsize=14, frameon=True, fancybox=True, shadow=True)
    ax.tick_params(axis='both', labelsize=14, width=2.5, length=6)
    ax.grid(True, alpha=0.3, linewidth=0.8)
    for spine in ax.spines.values():
        spine.set_linewidth(2)
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontweight('bold')
    plt.tight_layout()
    for fmt in ['png', 'svg', 'pdf']:
        plt.savefig(f'{filename}.{fmt}', dpi=600, bbox_inches='tight', format=fmt)
    plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
ax.plot(epochs, train_losses, markersize=5, linewidth=2.2, label='Train')
ax.plot(epochs, test_losses, markersize=5, linewidth=2.2, label='Test')
style_and_save(ax, 'Loss', 'loss_plot')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
ax.plot(epochs, train_accs, markersize=5, linewidth=2.2, label='Train')
ax.plot(epochs, test_accs, markersize=5, linewidth=2.2, label='Test')
style_and_save(ax, 'Pixel Accuracy', 'pixel_accuracy_plot')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
ax.plot(epochs, train_ious, markersize=5, linewidth=2.2, label='Train')
ax.plot(epochs, test_ious, markersize=5, linewidth=2.2, label='Test')
style_and_save(ax, 'IoU', 'iou_plot')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
ax.plot(epochs, train_dices, markersize=5, linewidth=2.2, label='Train')
ax.plot(epochs, test_dices, markersize=5, linewidth=2.2, label='Test')
style_and_save(ax, 'Dice / F1 Score', 'dice_f1_plot')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
ax.plot(epochs, train_precisions,markersize=5, linewidth=2.2, label='Train')
ax.plot(epochs, test_precisions, markersize=5, linewidth=2.2,label='Test')
style_and_save(ax, 'Precision', 'precision_plot')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
ax.plot(epochs, train_recalls, markersize=5, linewidth=2.2, label='Train')
ax.plot(epochs, test_recalls, markersize=5, linewidth=2.2, label='Test')
style_and_save(ax, 'Recall', 'recall_plot')

In [ ]:
# ============ BASELINE: Simple U-Net (fewer filters) ============
class UNetBaseline(nn.Module):
    def __init__(self):
        super().__init__()
        
        # Encoder (smaller)
        self.enc1 = self._block(3, 32)
        self.enc2 = self._block(32, 64)
        self.enc3 = self._block(64, 128)
        self.enc4 = self._block(128, 256)
        
        self.pool = nn.MaxPool2d(2)
        
        # Bottleneck
        self.bottleneck = self._block(256, 512)
        
        # Decoder
        self.up4 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec4 = self._block(512, 256)
        self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec3 = self._block(256, 128)
        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec2 = self._block(128, 64)
        self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec1 = self._block(64, 32)
        
        self.final = nn.Conv2d(32, 1, 1)
        self.sigmoid = nn.Sigmoid()
    
    def _block(self, in_ch, out_ch):
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    
    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        
        b = self.bottleneck(self.pool(e4))
        
        d4 = self.dec4(torch.cat([self.up4(b), e4], 1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], 1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], 1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], 1))
        
        return self.sigmoid(self.final(d1))

In [ ]:
import time

# ============ TRAIN BASELINE ============
baseline_model = UNetBaseline().to(device)
baseline_optimizer = torch.optim.Adam(baseline_model.parameters(), lr=1e-3)
baseline_criterion = FocalDiceLoss(alpha=0.75, gamma=2.0, dice_weight=0.5, focal_weight=0.5)
baseline_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(baseline_optimizer, patience=5, factor=0.5)

num_epochs_baseline = 50
best_baseline_iou = 0.0

# ⏱ Track training time
baseline_train_start = time.time()

for epoch in range(num_epochs_baseline):
    baseline_model.train()
    running_loss, running_iou = 0.0, 0.0
    total = 0
    
    for imgs, masks in train_loader:
        imgs, masks = imgs.to(device), masks.to(device)
        preds = baseline_model(imgs)
        loss = baseline_criterion(preds, masks)
        baseline_optimizer.zero_grad()
        loss.backward()
        baseline_optimizer.step()
        
        bs = imgs.size(0)
        running_loss += loss.item() * bs
        running_iou += iou_score(preds, masks).item() * bs
        total += bs
    
    # Eval
    baseline_model.eval()
    test_iou_sum, test_total = 0.0, 0
    with torch.no_grad():
        for imgs, masks in test_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            preds = baseline_model(imgs)
            bs = imgs.size(0)
            test_iou_sum += iou_score(preds, masks).item() * bs
            test_total += bs
    
    test_iou = test_iou_sum / test_total
    baseline_scheduler.step(running_loss / total)
    
    if test_iou > best_baseline_iou:
        best_baseline_iou = test_iou
        torch.save(baseline_model.state_dict(), 'baseline_model.pth')
    
    if (epoch + 1) % 10 == 0:
        print(f'Baseline Epoch [{epoch+1}/{num_epochs_baseline}] '
              f'Train IoU: {running_iou/total:.4f} | Test IoU: {test_iou:.4f}')

baseline_train_time = time.time() - baseline_train_start
print(f'\nBaseline Training Time: {baseline_train_time:.2f} sec')
print(f'Baseline Best Test IoU: {best_baseline_iou:.4f}')

In [ ]:
from sklearn.metrics import cohen_kappa_score
from scipy import stats
import numpy as np

# Load best models
baseline_model.load_state_dict(torch.load('baseline_model.pth'))
model.load_state_dict(torch.load('best_model.pth'))

baseline_model.eval()
model.eval()

# Per-sample metrics
baseline_ious, baseline_dices = [], []
proposed_ious, proposed_dices = [], []

# For Cohen's Kappa (pixel-level)
all_baseline_preds = []
all_proposed_preds = []
all_targets = []

with torch.no_grad():
    for imgs, masks in test_loader:
        imgs, masks = imgs.to(device), masks.to(device)
        
        # Baseline predictions
        base_preds = baseline_model(imgs)
        # Proposed model predictions
        prop_preds = model(imgs)
        
        # Per-sample IoU and Dice
        for j in range(imgs.size(0)):
            b_iou = iou_score(base_preds[j:j+1], masks[j:j+1]).item()
            b_dice = dice_score(base_preds[j:j+1], masks[j:j+1]).item()
            p_iou = iou_score(prop_preds[j:j+1], masks[j:j+1]).item()
            p_dice = dice_score(prop_preds[j:j+1], masks[j:j+1]).item()
            
            baseline_ious.append(b_iou)
            baseline_dices.append(b_dice)
            proposed_ious.append(p_iou)
            proposed_dices.append(p_dice)
            
            # Pixel-level predictions for Kappa
            base_bin = (base_preds[j] > 0.5).cpu().numpy().flatten()
            prop_bin = (prop_preds[j] > 0.5).cpu().numpy().flatten()
            target = masks[j].cpu().numpy().flatten()
            
            all_baseline_preds.extend(base_bin.tolist())
            all_proposed_preds.extend(prop_bin.tolist())
            all_targets.extend(target.tolist())

baseline_ious = np.array(baseline_ious)
baseline_dices = np.array(baseline_dices)
proposed_ious = np.array(proposed_ious)
proposed_dices = np.array(proposed_dices)

print(f"Collected {len(proposed_ious)} test samples")
print(f"Baseline → IoU: {baseline_ious.mean():.4f} | Dice: {baseline_dices.mean():.4f}")
print(f"Proposed → IoU: {proposed_ious.mean():.4f} | Dice: {proposed_dices.mean():.4f}")

In [ ]:
from scipy.stats import wilcoxon, ttest_rel, sem

# ============ 1. CONFIDENCE INTERVALS (95%) ============
def confidence_interval(data, confidence=0.95):
    n = len(data)
    mean = np.mean(data)
    se = sem(data)
    h = se * stats.t.ppf((1 + confidence) / 2, n - 1)
    return mean, mean - h, mean + h

# Baseline CIs
b_iou_mean, b_iou_lo, b_iou_hi = confidence_interval(baseline_ious)
b_dice_mean, b_dice_lo, b_dice_hi = confidence_interval(baseline_dices)

# Proposed CIs
p_iou_mean, p_iou_lo, p_iou_hi = confidence_interval(proposed_ious)
p_dice_mean, p_dice_lo, p_dice_hi = confidence_interval(proposed_dices)

print("=" * 70)
print("  95% CONFIDENCE INTERVALS")
print("=" * 70)
print(f"  Baseline U-Net → IoU: {b_iou_mean:.4f} [{b_iou_lo:.4f}, {b_iou_hi:.4f}]")
print(f"                 → Dice: {b_dice_mean:.4f} [{b_dice_lo:.4f}, {b_dice_hi:.4f}]")
print(f"  Proposed Model → IoU: {p_iou_mean:.4f} [{p_iou_lo:.4f}, {p_iou_hi:.4f}]")
print(f"                 → Dice: {p_dice_mean:.4f} [{p_dice_lo:.4f}, {p_dice_hi:.4f}]")


# ============ 2. COHEN'S KAPPA ============
baseline_kappa = cohen_kappa_score(all_targets, all_baseline_preds)
proposed_kappa = cohen_kappa_score(all_targets, all_proposed_preds)

print("\n" + "=" * 70)
print("  COHEN'S KAPPA (Pixel-Level Agreement)")
print("=" * 70)
print(f"  Baseline U-Net → κ = {baseline_kappa:.4f}")
print(f"  Proposed Model → κ = {proposed_kappa:.4f}")


# ============ 3. PAIRED T-TEST ============
t_stat_iou, p_value_ttest_iou = ttest_rel(proposed_ious, baseline_ious)
t_stat_dice, p_value_ttest_dice = ttest_rel(proposed_dices, baseline_dices)

print("\n" + "=" * 70)
print("  PAIRED T-TEST (Proposed vs Baseline)")
print("=" * 70)
print(f"  IoU  → t = {t_stat_iou:.4f}, p-value = {p_value_ttest_iou:.6f} {'✅ Significant' if p_value_ttest_iou < 0.05 else '❌ Not Significant'}")
print(f"  Dice → t = {t_stat_dice:.4f}, p-value = {p_value_ttest_dice:.6f} {'✅ Significant' if p_value_ttest_dice < 0.05 else '❌ Not Significant'}")


# ============ 4. WILCOXON SIGNED-RANK TEST ============
w_stat_iou, p_value_wilcox_iou = wilcoxon(proposed_ious, baseline_ious)
w_stat_dice, p_value_wilcox_dice = wilcoxon(proposed_dices, baseline_dices)

print("\n" + "=" * 70)
print("  WILCOXON SIGNED-RANK TEST (Proposed vs Baseline)")
print("=" * 70)
print(f"  IoU  → W = {w_stat_iou:.4f}, p-value = {p_value_wilcox_iou:.6f} {'✅ Significant' if p_value_wilcox_iou < 0.05 else '❌ Not Significant'}")
print(f"  Dice → W = {w_stat_dice:.4f}, p-value = {p_value_wilcox_dice:.6f} {'✅ Significant' if p_value_wilcox_dice < 0.05 else '❌ Not Significant'}")

In [ ]:
pip install thop

In [ ]:
import time
import torch
from thop import profile, clever_format

# Install thop if needed: !pip install thop

# ============ MODEL PARAMETERS ============
def count_parameters(model):
    return sum(p.numel() for p in model.parameters()) / 1e6  # in Millions

baseline_params = count_parameters(baseline_model)
proposed_params = count_parameters(model)

# ============ FLOPs ============
dummy_input = torch.randn(1, 3, 256, 256).to(device)

baseline_model.eval()
model.eval()

base_flops, _ = profile(baseline_model, inputs=(dummy_input,), verbose=False)
prop_flops, _ = profile(model, inputs=(dummy_input,), verbose=False)

base_flops_g = base_flops / 1e9  # Convert to GFLOPs
prop_flops_g = prop_flops / 1e9

# ============ INFERENCE TIME ============
def measure_inference_time(model, dataloader, device, num_runs=3):
    model.eval()
    times = []
    with torch.no_grad():
        for run in range(num_runs):
            start = time.time()
            for imgs, masks in dataloader:
                imgs = imgs.to(device)
                _ = model(imgs)
            elapsed = time.time() - start
            times.append(elapsed)
    total_samples = len(dataloader.dataset)
    avg_total_time = np.mean(times)
    per_image_time = avg_total_time / total_samples
    return avg_total_time, per_image_time

base_total_time, base_per_img = measure_inference_time(baseline_model, test_loader, device)
prop_total_time, prop_per_img = measure_inference_time(model, test_loader, device)

# ============ GPU MEMORY USAGE ============
def measure_memory(model, input_tensor):
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.empty_cache()
    model.eval()
    with torch.no_grad():
        _ = model(input_tensor)
    peak_memory = torch.cuda.max_memory_allocated() / (1024 ** 2)  # MB
    return peak_memory

dummy = torch.randn(1, 3, 256, 256).to(device)
base_memory = measure_memory(baseline_model, dummy)
prop_memory = measure_memory(model, dummy)

# ============ TRAINING TIME ============
# baseline_train_time already recorded above
# For proposed model, if you didn't record it, estimate:
# proposed_train_time = <your recorded time>
# If not recorded, you can note it manually

print("\n" + "=" * 80)
print("  COMPUTATIONAL COST COMPARISON")
print("=" * 80)
print(f"{'Metric':<25} {'Baseline U-Net':>18} {'Proposed Model':>18}")
print("-" * 65)
print(f"{'Parameters (M)':<25} {baseline_params:>18.2f} {proposed_params:>18.2f}")
print(f"{'FLOPs (G)':<25} {base_flops_g:>18.2f} {prop_flops_g:>18.2f}")
print(f"{'Training Time (Sec)':<25} {baseline_train_time:>18.2f} {'<your_time>':>18}")
print(f"{'Inference Time/Img (Sec)':<25} {base_per_img:>18.4f} {prop_per_img:>18.4f}")
print(f"{'Peak Memory (MB)':<25} {base_memory:>18.2f} {prop_memory:>18.2f}")
print("=" * 80)

In [ ]:
import pandas as pd

# ============ STATISTICAL ANALYSIS TABLE ============
stat_data = {
    'Method': ['Baseline U-Net', 'Proposed Model'],
    'Dataset': ['ATLDSD', 'ATLDSD'],
    '95% CI (IoU)': [f'{b_iou_mean:.4f} [{b_iou_lo:.4f}-{b_iou_hi:.4f}]',
                     f'{p_iou_mean:.4f} [{p_iou_lo:.4f}-{p_iou_hi:.4f}]'],
    '95% CI (Dice)': [f'{b_dice_mean:.4f} [{b_dice_lo:.4f}-{b_dice_hi:.4f}]',
                      f'{p_dice_mean:.4f} [{p_dice_lo:.4f}-{p_dice_hi:.4f}]'],
    'Wilcoxon p-value': ['—', f'{p_value_wilcox_iou:.6f}'],
    'Paired t-test p-value': ['—', f'{p_value_ttest_iou:.6f}'],
    "Cohen's κ": [f'{baseline_kappa:.4f}', f'{proposed_kappa:.4f}'],
}

stat_df = pd.DataFrame(stat_data)
print("\n📊 STATISTICAL ANALYSIS")
print(stat_df.to_string(index=False))

# ============ COMPUTATIONAL COST TABLE ============
comp_data = {
    'Method': ['Baseline U-Net', 'Proposed Model'],
    'Dataset': ['ATLDSD', 'ATLDSD'],
    'Training Time (Sec)': [f'{baseline_train_time:.2f}', '<your_time>'],
    'Inference Time (Sec)': [f'{base_per_img:.4f}', f'{prop_per_img:.4f}'],
    'Memory (MB)': [f'{base_memory:.2f}', f'{prop_memory:.2f}'],
    'FLOPs (G)': [f'{base_flops_g:.2f}', f'{prop_flops_g:.2f}'],
    'Parameters (M)': [f'{baseline_params:.2f}', f'{proposed_params:.2f}'],
}

comp_df = pd.DataFrame(comp_data)
print("\n💻 COMPUTATIONAL COST")
print(comp_df.to_string(index=False))